## MIMIC-IV + eICU Multi-Center Validation

Pipeline structure:
- **Steps 0-4**: Data download, cohort building, vital sign extraction, feature engineering
- **Step 5**: Train/val/test split
- **Step 6**: A(t) calibration on TRAIN ONLY (recalibrated weights, population statistics, hourly A(t))
- **Step 7**: XGBoost on 4 conditions (Full, NEWS2-set, CAS-4, CAS-4 hybrid A)
- **Step 8**: NEWS2 scoring baseline
- **Step 9**: Feature reduction curve (9 sensor levels) — central analysis
- **Step 10**: Additional models (CatBoost, Random Forest)
- **Step 11**: SHAP feature importance + ranking CSV
- **Step 12**: Calibration analysis (slope, CITL, Hosmer-Lemeshow)
- **Step 13**: Decision Curve Analysis with explicit NB at p_t=0.10
- **Step 14**: Subgroup analysis
- **Step 15**: eICU multi-center external validation (hourly A(t), 138 hospitals)
- **Step 16**: Save all results to JSON
- **Step 17**: Bootstrap pairwise AUROC comparison (n=2000)

All results → `~/mimic4_study/results/all_results.json` for Word template.

### v4 changelog (May 2026)
- A(t) weights and population statistics now computed on **training set only**
  (previously on full cohort — addressed test-set leakage concern).
- eICU A(t) now computed **hourly** matching MIMIC-IV methodology
  (previously simplified to `at_max=at_mean`, `at_std=0`).
- Bootstrap permutation test now executes as part of the pipeline.
- SHAP feature ranking exported to CSV alongside summary plot.
- DCA reports explicit net benefit at p_t=0.10 for each model.
- All paths use `os.path.expanduser` for portability.

### v5 changelog (May 2026, reviewer-driven)

Additional analyses inserted as numbered sub-steps to keep the original Step IDs stable:
- **Step 9b**: GCS / consciousness column diagnostic — explains the L3-NEWS2 vs. L4-5vitals identity in Table II.
- **Step 12b**: Decile-level observed/expected diagnostic for top-3 models — supplements the Hosmer–Lemeshow test, which is over-sensitive at large n.
- **Step 12c**: L7-2vitals calibration sanity check — bootstraps the slope=2.654 estimate, plots predicted-probability distributions for L6/L7/L8.
- **Step 13b**: Platt scaling on validation set + recomputed DCA — empirically tests the recalibration claim against NEWS2 scoring at p_t=0.10.
- **Step 15b**: Per-hospital characterization of included (n=138) vs. excluded eICU hospitals — quantifies direction of selection bias.
- **Step 17**: Bootstrap pairwise comparison now includes the central L6 vs. L7 knee-point test (formal p-value, not descriptive CI overlap).

These additions do not modify the original Steps 0–17; they only add diagnostic output. Re-running the original pipeline (Restart & Run All) is still the correct way to reproduce the manuscript's headline numbers.


---
## Step 0: Setup & Installation

In [ ]:
# Install required packages (uncomment if needed, or run: pip install -r requirements.txt)
# !pip install pandas numpy pyarrow scikit-learn xgboost catboost shap matplotlib scipy tqdm wfdb requests

In [ ]:
import os
import sys
import json
import warnings
from dotenv import load_dotenv

# Load credentials and optional config from a local .env file
# (PHYSIONET_USER, PHYSIONET_PASSWORD, CAS_BASE_DIR). Existing
# environment variables take precedence and are never overwritten.
load_dotenv()

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, brier_score_loss, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# PhysioNet credentials — set via environment variable or edit here
PHYSIONET_USER = os.environ.get("PHYSIONET_USER", "")
if not PHYSIONET_USER:
    raise RuntimeError(
        "PHYSIONET_USER is not set. Either:\n"
        "  1. Create a .env file in the project root with: PHYSIONET_USER=your_username\n"
        "     (see .env.example for a template), OR\n"
        "  2. Export it in your shell: export PHYSIONET_USER=your_username\n"
    )

MIMIC_PACKAGE = "mimiciv"
MIMIC_VER = "3.1"

# All paths use expanduser for portability across machines
BASE_DIR = os.path.expanduser(os.environ.get("CAS_BASE_DIR", "~/mimic4_study"))
RAW_DIR = os.path.join(BASE_DIR, "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")

RANDOM_STATE = 42

for d in [BASE_DIR, RAW_DIR, PROCESSED_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Base directory: {BASE_DIR}")

assert PHYSIONET_USER and PHYSIONET_USER != "YOUR_PHYSIONET_USERNAME", \
    "Please set PHYSIONET_USER environment variable or edit this cell."

# A(t) clinical severity multipliers (fixed, from prior work [14])
# Weights are recalibrated empirically on training set in Step 6.
AT_MULTIPLIERS = {'resp_rate': 3.0, 'spo2': 2.0, 'heart_rate': 1.0, 'temperature': 1.0}

In [ ]:
# MIMIC-IV itemids for vital signs
VITAL_SIGN_ITEMS = {
    "heart_rate":   [220045],
    "spo2":         [220277],
    "resp_rate":    [220210, 224690],
    "temperature":  [223761, 223762],   # Fahrenheit and Celsius
    "sbp":          [220179, 220050],
    "dbp":          [220180, 220051],
    "map":          [220052, 220181, 225312],
    # GCS in MIMIC-IV: no composite itemid in chartevents. Components are
    # extracted separately and gcs_total is reconstructed in Cell 16.
    # Source: MIT-LCP/mimic-code (mimic-iv/concepts/firstday/gcs.sql).
    "gcs_eye":      [220739],
    "gcs_verbal":   [223900],
    "gcs_motor":    [223901],
}

ALL_ITEMIDS = [iid for ids in VITAL_SIGN_ITEMS.values() for iid in ids]
ITEMID_TO_VITAL = {iid: v for v, ids in VITAL_SIGN_ITEMS.items() for iid in ids}

CORE_VITALS = ["heart_rate", "spo2", "resp_rate", "temperature"]

# Plausible ranges for data cleaning
PLAUSIBLE_RANGES = {
    "heart_rate": (20, 300),
    "spo2": (50, 100),
    "resp_rate": (2, 70),
    "temperature": (30, 42),
    "sbp": (40, 300),
    "dbp": (20, 200),
    "map": (30, 250),
    "gcs_eye":    (1, 4),
    "gcs_verbal": (1, 5),
    "gcs_motor":  (1, 6),
    "gcs_total":  (3, 15),  # synthetic, validated by reconstruction in Cell 16
}

# Suffixes for temporal statistical features (defined once, used everywhere)
SUFFIXES = ["_mean", "_std", "_min", "_max", "_first", "_last", "_range", "_cv", "_trend"]

print(f"Core vitals: {CORE_VITALS}")
print(f"Total itemids to extract: {len(ALL_ITEMIDS)}")

---
## FAST START: Load Cached Data

If you already ran the original pipeline and have processed files,
this cell loads them and skips Steps 1-4 (data extraction).

A(t) recalibration in Step 6 will still run on training set.

In [ ]:
# Check for cached data from previous run
SKIP_EXTRACTION = False

_cohort_path = os.path.join(PROCESSED_DIR, 'cohort_final.csv')
_features_path = os.path.join(PROCESSED_DIR, 'features_xgboost.csv')
_hourly_path = os.path.join(PROCESSED_DIR, 'hourly_series.csv.gz')

if os.path.exists(_cohort_path) and os.path.exists(_features_path) and os.path.exists(_hourly_path):
    print('Found cached data from previous run:')
    cohort = pd.read_csv(_cohort_path)
    features = pd.read_csv(_features_path)
    hourly_full = pd.read_csv(_hourly_path, compression='gzip')

    # Drop any A(t) columns from cached features — they will be RECOMPUTED in Step 6
    # using train-only weights (v4 fix for data leakage).
    at_cols_to_drop = [c for c in ['at_mean', 'at_max', 'at_std', 'at_last'] if c in features.columns]
    if at_cols_to_drop:
        features = features.drop(columns=at_cols_to_drop)
        print(f'  Dropped cached A(t) columns: {at_cols_to_drop} (will be recomputed train-only)')

    # Detect target column
    if 'in_hospital_mortality' not in cohort.columns and 'hospital_expire_flag' in cohort.columns:
        cohort['in_hospital_mortality'] = cohort['hospital_expire_flag']
    if 'in_hospital_mortality' not in features.columns and 'in_hospital_mortality' in cohort.columns:
        features = features.merge(cohort[['stay_id','in_hospital_mortality']], on='stay_id', how='left')

    # Ensure age/gender in features
    for col in ['age', 'gender']:
        if col not in features.columns and col in cohort.columns:
            features = features.merge(cohort[['stay_id', col]], on='stay_id', how='left')
    if 'gender_binary' not in features.columns and 'gender' in features.columns:
        features['gender_binary'] = (features['gender'] == 'M').astype(int)

    print(f'  Cohort: {len(cohort)} stays')
    print(f'  Features: {features.shape}')
    print(f'  Hourly grid: {len(hourly_full)} rows')
    print(f'  Mortality: {cohort["in_hospital_mortality"].sum()} ({cohort["in_hospital_mortality"].mean():.1%})')

    SKIP_EXTRACTION = True
    print('\n>>> Steps 1-4 will be SKIPPED. Jump to Step 5 (train/val/test split).')
else:
    print(f'No cached data found in {PROCESSED_DIR}')
    print('Will run full extraction pipeline (Steps 1-4).')

---
## Step 1: Download MIMIC-IV Tables from PhysioNet

We download only the tables we need, not the entire database.

**Note:** `chartevents.csv.gz` is ~6 GB. This step may take 30-60 minutes.

Set environment variable `PHYSIONET_PASSWORD` to avoid interactive prompt.

In [ ]:
if SKIP_EXTRACTION:
    print('Cached data available. Skipping download.')
    # Still need session for eICU later (Step 15) — will prompt at that point if needed.
    session = None
else:
    import requests
    import getpass

    pw_env = os.environ.get("PHYSIONET_PASSWORD")
    if pw_env:
        password = pw_env
        print(f"Using PHYSIONET_PASSWORD from environment for {PHYSIONET_USER}")
    else:
        password = getpass.getpass(f"PhysioNet password for {PHYSIONET_USER}: ")

    session = requests.Session()
    login_url = "https://physionet.org/login/"
    resp = session.get(login_url)
    csrf = session.cookies.get("csrftoken", "")

    login_data = {"username": PHYSIONET_USER, "password": password,
                  "csrfmiddlewaretoken": csrf}
    resp = session.post(login_url, data=login_data, headers={"Referer": login_url})
    print(f"Login status: {resp.status_code}")

    REQUIRED_TABLES = {
        "hosp": ["admissions", "patients"],
        "icu":  ["icustays", "chartevents", "d_items"],
    }
    EXPECTED_SIZES = {  # MB, used to detect resumable/incomplete downloads
        "admissions": 10, "patients": 0.3, "icustays": 2,
        "chartevents": 5000, "d_items": 0.05,
    }
    base_url = f"https://physionet.org/files/{MIMIC_PACKAGE}/{MIMIC_VER}"

    for module, tables in REQUIRED_TABLES.items():
        for table in tables:
            filename = f"{table}.csv.gz"
            url = f"{base_url}/{module}/{filename}"
            outpath = os.path.join(RAW_DIR, module)
            os.makedirs(outpath, exist_ok=True)
            filepath = os.path.join(outpath, filename)

            existing_size = os.path.getsize(filepath) if os.path.exists(filepath) else 0
            min_expected = EXPECTED_SIZES.get(table, 0) * 0.8 * 1e6

            if existing_size > min_expected and min_expected > 0:
                print(f"  [SKIP] {module}/{filename} ({existing_size/1e6:.0f} MB)")
                continue

            print(f"  [DOWNLOAD] {module}/{filename} ...", end=" ", flush=True)
            try:
                headers_req = {}
                if existing_size > 0:
                    headers_req["Range"] = f"bytes={existing_size}-"
                resp = session.get(url, stream=True, headers=headers_req)
                if resp.status_code == 416:
                    print(f"\r  [DONE] {module}/{filename} ({existing_size/1e6:.0f} MB) - already complete")
                    continue
                resp.raise_for_status()

                if resp.status_code == 206:
                    total = existing_size + int(resp.headers.get("content-length", 0))
                    mode, downloaded = "ab", existing_size
                else:
                    total = int(resp.headers.get("content-length", 0))
                    mode, downloaded = "wb", 0

                with open(filepath, mode) as f:
                    for chunk in resp.iter_content(chunk_size=1024*1024):
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total > 0:
                            pct = downloaded / total * 100
                            print(f"\r  [DOWNLOAD] {module}/{filename} ... {pct:.0f}% "
                                  f"({downloaded/1e6:.0f}/{total/1e6:.0f} MB)", end="", flush=True)
                print(f"\r  [DONE] {module}/{filename} ({os.path.getsize(filepath)/1e6:.0f} MB)" + " "*30)
            except Exception as e:
                print(f"\n  [ERROR] {e}")

---
## Step 2: Extract Study Cohort

**Inclusion criteria:**
- Adult (age ≥ 18)
- At least 1 ICU stay ≥ 24 hours
- First ICU stay per patient only

**Outcome:** In-hospital mortality (binary)

In [ ]:
if SKIP_EXTRACTION:
    print('Skipping cohort extraction (cached).')
else:
    print("Loading tables...")
    patients = pd.read_csv(os.path.join(RAW_DIR, "hosp", "patients.csv.gz"), compression="gzip")
    admissions = pd.read_csv(
        os.path.join(RAW_DIR, "hosp", "admissions.csv.gz"), compression="gzip",
        parse_dates=["admittime", "dischtime", "deathtime"]
    )
    icustays = pd.read_csv(
        os.path.join(RAW_DIR, "icu", "icustays.csv.gz"), compression="gzip",
        parse_dates=["intime", "outtime"]
    )

    print(f"  Patients: {len(patients)}")
    print(f"  Admissions: {len(admissions)}")
    print(f"  ICU stays: {len(icustays)}")

In [ ]:
if not SKIP_EXTRACTION:
    # Merge
    cohort = icustays.merge(patients, on="subject_id", how="left")
    cohort = cohort.merge(admissions, on=["subject_id", "hadm_id"], how="left")

    # Compute age
    cohort["admit_year"] = cohort["admittime"].dt.year
    cohort["age"] = cohort["anchor_age"] + (cohort["admit_year"] - cohort["anchor_year"])

    # ICU length of stay (hours)
    cohort["icu_los_hours"] = (cohort["outtime"] - cohort["intime"]).dt.total_seconds() / 3600

    # In-hospital mortality
    cohort["in_hospital_mortality"] = (
        cohort["deathtime"].notna() & (cohort["deathtime"] <= cohort["dischtime"])
    ).astype(int)

    print(f"Total ICU stays before filtering: {len(cohort)}")

In [ ]:
if not SKIP_EXTRACTION:
    print("Applying criteria:")

    n = len(cohort)
    cohort = cohort[cohort["age"] >= 18]
    print(f"  Age >= 18: {n} -> {len(cohort)} (removed {n - len(cohort)})")

    n = len(cohort)
    cohort = cohort[cohort["icu_los_hours"] >= 24]
    print(f"  LOS >= 24h: {n} -> {len(cohort)} (removed {n - len(cohort)})")

    n = len(cohort)
    cohort = cohort.sort_values("intime").groupby("subject_id").first().reset_index()
    print(f"  First stay: {n} -> {len(cohort)} (removed {n - len(cohort)})")

    cohort_cols = ["subject_id", "hadm_id", "stay_id", "intime", "outtime",
                   "age", "gender", "icu_los_hours", "in_hospital_mortality"]
    cohort = cohort[cohort_cols].copy()
    cohort.to_csv(os.path.join(PROCESSED_DIR, "cohort.csv"), index=False)

    print(f"\nCohort: {len(cohort)} patients")
    print(f"Mortality: {cohort['in_hospital_mortality'].sum()} ({cohort['in_hospital_mortality'].mean()*100:.1f}%)")
    print(f"Age: {cohort['age'].median():.0f} (IQR {cohort['age'].quantile(.25):.0f}-{cohort['age'].quantile(.75):.0f})")
    print(f"Male: {(cohort['gender']=='M').sum()} ({(cohort['gender']=='M').mean()*100:.1f}%)")

---
## Step 3: Extract Vital Signs from chartevents

This is the slowest step. `chartevents` is processed in 5M-row chunks.

**Expected time:** 20-40 minutes

In [ ]:
if not SKIP_EXTRACTION:
    stay_ids = set(cohort["stay_id"].values)
    intime_map = dict(zip(cohort["stay_id"], pd.to_datetime(cohort["intime"])))

    chartevents_path = os.path.join(RAW_DIR, "icu", "chartevents.csv.gz")
    assert os.path.exists(chartevents_path), f"File not found: {chartevents_path}"

    all_vitals = []
    chunk_size = 5_000_000

    reader = pd.read_csv(
        chartevents_path, compression="gzip", chunksize=chunk_size,
        usecols=["stay_id", "itemid", "charttime", "valuenum"],
        dtype={"stay_id": "Int64", "itemid": "Int64", "valuenum": float},
        parse_dates=["charttime"],
    )

    for chunk in tqdm(reader, desc="Processing chartevents"):
        mask = (
            chunk["stay_id"].isin(stay_ids) &
            chunk["itemid"].isin(ALL_ITEMIDS) &
            chunk["valuenum"].notna()
        )
        filtered = chunk[mask].copy()
        if len(filtered) == 0:
            continue

        filtered["vital"] = filtered["itemid"].map(ITEMID_TO_VITAL)
        filtered["intime"] = filtered["stay_id"].map(intime_map)
        filtered["hours_since_admit"] = (
            (filtered["charttime"] - filtered["intime"]).dt.total_seconds() / 3600
        )
        filtered = filtered[
            (filtered["hours_since_admit"] >= 0) &
            (filtered["hours_since_admit"] < 24)
        ]
        if len(filtered) > 0:
            all_vitals.append(filtered[["stay_id", "vital", "charttime", "hours_since_admit", "valuenum"]])

    vitals = pd.concat(all_vitals, ignore_index=True)
    print(f"\nTotal vital sign measurements: {len(vitals):,}")

In [ ]:
if not SKIP_EXTRACTION:
    # Temperature: convert Fahrenheit to Celsius
    mask_f = (vitals["vital"] == "temperature") & (vitals["valuenum"] > 50)
    vitals.loc[mask_f, "valuenum"] = (vitals.loc[mask_f, "valuenum"] - 32) * 5 / 9
    print(f"Converted {mask_f.sum()} temperature values from F to C")

    before = len(vitals)
    for vital_name, (lo, hi) in PLAUSIBLE_RANGES.items():
        bad = (vitals["vital"] == vital_name) & (
            (vitals["valuenum"] < lo) | (vitals["valuenum"] > hi)
        )
        vitals = vitals[~bad]
    print(f"Removed {before - len(vitals)} implausible values")


    # ---- Reconstruct gcs_total from its three components ----
    # MIMIC-IV does not store a composite GCS in chartevents. Clinical
    # practice charts the three components together, so we pivot on
    # (stay_id, charttime), require all three present, and sum them.
    # Synthetic gcs_total rows are appended back into the long-format vitals.
    gcs_components = ["gcs_eye", "gcs_verbal", "gcs_motor"]
    gcs_long = vitals[vitals["vital"].isin(gcs_components)]
    if len(gcs_long) > 0:
        gcs_wide = gcs_long.pivot_table(
            index=["stay_id", "charttime", "hours_since_admit"],
            columns="vital", values="valuenum", aggfunc="mean",
        ).reset_index()
        gcs_wide = gcs_wide.dropna(subset=gcs_components)
        gcs_wide["valuenum"] = (
            gcs_wide["gcs_eye"] + gcs_wide["gcs_verbal"] + gcs_wide["gcs_motor"]
        )
        gcs_wide["vital"] = "gcs_total"
        lo, hi = PLAUSIBLE_RANGES["gcs_total"]
        gcs_wide = gcs_wide[(gcs_wide["valuenum"] >= lo) & (gcs_wide["valuenum"] <= hi)]
        gcs_total_rows = gcs_wide[
            ["stay_id", "vital", "charttime", "hours_since_admit", "valuenum"]
        ]
        vitals = pd.concat([vitals, gcs_total_rows], ignore_index=True)
        print(f"Reconstructed {len(gcs_total_rows)} gcs_total rows from components")
    else:
        print("No GCS component rows found - gcs_total cannot be reconstructed")

    vitals.to_csv(os.path.join(PROCESSED_DIR, "vitals_raw.csv.gz"), index=False, compression="gzip")
    print(f"Saved: {os.path.join(PROCESSED_DIR, 'vitals_raw.csv.gz')}")

    print("\nVital sign counts:")
    print(vitals["vital"].value_counts().to_string())

---
## Step 4: Create Features

1. Resample to hourly intervals (0-23 hours)
2. Forward-fill gaps ≤ 3 hours
3. Compute 9 temporal statistical features per vital sign
4. Filter cohort: require all 4 core vitals with ≥ 6 non-null hours

In [ ]:
if not SKIP_EXTRACTION:
    vitals["hour"] = vitals["hours_since_admit"].astype(int).clip(0, 23)

    hourly = (
        vitals.groupby(["stay_id", "vital", "hour"])["valuenum"]
        .mean()
        .reset_index()
    )

    hourly_pivot = hourly.pivot_table(
        index=["stay_id", "hour"], columns="vital", values="valuenum"
    ).reset_index()

    all_hours = pd.DataFrame({
        "hour": list(range(24)) * len(cohort),
        "stay_id": np.repeat(cohort["stay_id"].values, 24)
    })
    hourly_full = all_hours.merge(hourly_pivot, on=["stay_id", "hour"], how="left")
    hourly_full = hourly_full.sort_values(["stay_id", "hour"])

    for col in hourly_full.columns:
        if col in ["stay_id", "hour"]:
            continue
        hourly_full[col] = (
            hourly_full.groupby("stay_id")[col]
            .transform(lambda s: s.fillna(method="ffill", limit=3))
        )

    print(f"Hourly grid: {len(hourly_full)} rows ({len(hourly_full)//24} patients x 24 hours)")

In [ ]:
if not SKIP_EXTRACTION:
    # Filter: require all 4 core vitals with >= 6 non-null hours
    completeness = (
        hourly_full.groupby("stay_id")[CORE_VITALS]
        .apply(lambda df: (df.notna().sum() >= 6).all())
    )
    valid_stays = completeness[completeness].index.tolist()

    cohort = cohort[cohort["stay_id"].isin(valid_stays)].copy()
    hourly_full = hourly_full[hourly_full["stay_id"].isin(valid_stays)].copy()

    print(f"Patients with complete core vitals: {len(cohort)}")
    print(f"Mortality rate: {cohort['in_hospital_mortality'].mean():.3f}")

In [ ]:
if not SKIP_EXTRACTION:
    # Iterate over every vital column in hourly_full (not just VITAL_SIGN_ITEMS.keys()),
    # so reconstructed columns like gcs_total are included.
    all_vitals_names = [c for c in hourly_full.columns if c not in {'stay_id', 'hour'}]

    feature_rows = []
    for stay_id, group in tqdm(hourly_full.groupby("stay_id"), desc="Computing features"):
        row = {"stay_id": stay_id}
        for vital in all_vitals_names:
            vals = group[vital].dropna()
            if len(vals) == 0:
                continue
            row[f"{vital}_mean"] = vals.mean()
            row[f"{vital}_std"] = vals.std() if len(vals) > 1 else 0
            row[f"{vital}_min"] = vals.min()
            row[f"{vital}_max"] = vals.max()
            row[f"{vital}_first"] = vals.iloc[0]
            row[f"{vital}_last"] = vals.iloc[-1]
            row[f"{vital}_range"] = vals.max() - vals.min()
            row[f"{vital}_cv"] = (vals.std() / vals.mean()) if vals.mean() != 0 else 0
            if len(vals) >= 2:
                row[f"{vital}_trend"] = np.polyfit(np.arange(len(vals)), vals.values, 1)[0]
            else:
                row[f"{vital}_trend"] = 0.0
        feature_rows.append(row)

    features = pd.DataFrame(feature_rows)
    features = features.merge(
        cohort[["stay_id", "age", "gender", "in_hospital_mortality"]], on="stay_id"
    )
    features["gender_binary"] = (features["gender"] == "M").astype(int)

    print(f"\nFeature matrix: {features.shape[0]} patients x {features.shape[1]} columns")

In [ ]:
if not SKIP_EXTRACTION:
    cohort.to_csv(os.path.join(PROCESSED_DIR, "cohort_final.csv"), index=False)
    features.to_csv(os.path.join(PROCESSED_DIR, "features_xgboost.csv"), index=False)
    hourly_full.to_csv(os.path.join(PROCESSED_DIR, "hourly_series.csv.gz"), index=False, compression="gzip")

    print("Saved:")
    print(f"  cohort_final.csv ({len(cohort)} patients)")
    print(f"  features_xgboost.csv ({features.shape})")
    print(f"  hourly_series.csv.gz ({len(hourly_full)} rows)")

---
## Step 5: Train/Val/Test Split (70/15/15, stratified)

**v4 fix:** This split is now performed BEFORE A(t) recalibration, so that
weights and population statistics are computed on training data only,
preventing test-set leakage.

In [ ]:
target = "in_hospital_mortality"
y = features[target]

train_idx, temp_idx = train_test_split(
    features.index, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y.loc[temp_idx], random_state=RANDOM_STATE
)

# Sort indices for reproducibility downstream
train_idx = np.sort(train_idx); val_idx = np.sort(val_idx); test_idx = np.sort(test_idx)

print(f"Train: {len(train_idx)} (mortality {y.loc[train_idx].mean():.3f})")
print(f"Val:   {len(val_idx)} (mortality {y.loc[val_idx].mean():.3f})")
print(f"Test:  {len(test_idx)} (mortality {y.loc[test_idx].mean():.3f})")

# Build set of train stay_ids for downstream A(t) calibration
train_stay_ids = set(features.loc[train_idx, 'stay_id'].values)
print(f"Train stay_ids: {len(train_stay_ids)} unique")

---
## Step 6: Compute A(t) Index — Calibrated on TRAINING SET ONLY

From [14]: $A(t) = \text{sat}\left(\sum_i w_i \cdot m_i \cdot \varphi(z_i(t))\right)$

**v4 methodology fix:** Weights $w_i$ and population statistics
($\mu_i, \sigma_i$) for z-score normalization are now computed
exclusively on the **training set** to prevent test-set leakage.
Hourly A(t) is then evaluated for ALL patients (train, val, test, eICU)
using these training-derived parameters.

In [ ]:
# Step 6.1: Correlation-derived weights from TRAINING SET ONLY
print("Correlation with in-hospital mortality (TRAINING SET):")
correlations = {}
train_features = features.loc[train_idx]

for vital in CORE_VITALS:
    col = f"{vital}_mean"
    if col in train_features.columns:
        r = train_features[col].corr(train_features["in_hospital_mortality"])
        correlations[vital] = abs(r)
        print(f"  |r({vital})| = {abs(r):.4f}")

total_r = sum(correlations.values())
weights = {v: r / total_r for v, r in correlations.items()}
print(f"\nNormalized weights (train-only): {json.dumps({k: round(v, 4) for k, v in weights.items()}, indent=2)}")
print(f"Sum of weights: {sum(weights.values()):.4f}")

# Compatibility: align global AT_WEIGHTS with newly computed weights
AT_WEIGHTS = dict(weights)
print(f"\nAT_WEIGHTS updated: {AT_WEIGHTS}")

In [ ]:
# Step 6.2: Population statistics from TRAINING SET ONLY
pop_stats = {}
for vital in CORE_VITALS:
    col = f"{vital}_mean"
    if col in train_features.columns:
        pop_stats[vital] = {
            "mean": float(train_features[col].mean()),
            "std":  float(train_features[col].std()),
            # Keep legacy keys for backward compatibility
            "mu":    float(train_features[col].mean()),
            "sigma": float(train_features[col].std()),
        }
        print(f"  {vital}: mean={pop_stats[vital]['mean']:.2f}, std={pop_stats[vital]['std']:.2f}")

# Multipliers from [14]
multipliers = dict(AT_MULTIPLIERS)

In [ ]:
# Step 6.3: Compute hourly A(t) for ALL patients using train-derived weights/stats
def phi(z):
    """Piecewise-linear sensitivity function from [14]"""
    az = abs(z)
    if az <= 0.5:
        return 0.0
    elif az < 2.0:
        return (az - 0.5) / 1.5
    else:
        return 1.0

def compute_at_hourly(hourly_df, weights, pop_stats, multipliers, core_vitals, id_col=None):
    """Compute hourly A(t) for a long-format hourly dataframe.

    Vectorized version: ~50x faster than a row-by-row loop.

    The id column name is auto-detected from common conventions:
      - 'stay_id'             (MIMIC-IV)
      - 'patientunitstayid'   (eICU)
    or can be passed explicitly via the id_col argument.
    """
    if id_col is None:
        if 'stay_id' in hourly_df.columns:
            id_col = 'stay_id'
        elif 'patientunitstayid' in hourly_df.columns:
            id_col = 'patientunitstayid'
        else:
            raise ValueError(
                f"compute_at_hourly: cannot auto-detect id column. "
                f"Pass id_col= explicitly. Available columns: {list(hourly_df.columns)}"
            )

    keep_cols = [id_col, "hour"] + [v for v in core_vitals if v in hourly_df.columns]
    df = hourly_df[keep_cols].copy()
    at_total = np.zeros(len(df))
    for vital in core_vitals:
        if vital not in df.columns:
            continue
        mu = pop_stats[vital]["mean"]
        sigma = pop_stats[vital]["std"]
        z = (df[vital].values - mu) / sigma
        az = np.abs(z)
        # Piecewise-linear phi, vectorized
        phi_z = np.where(np.isnan(az), 0.0,
                np.where(az <= 0.5, 0.0,
                np.where(az < 2.0, (az - 0.5) / 1.5, 1.0)))
        at_total = at_total + weights[vital] * multipliers[vital] * phi_z
    df["at_value"] = np.clip(at_total, 0.0, 1.0)
    return df[[id_col, "hour", "at_value"]]

print("Computing hourly A(t) for full cohort (train weights applied to all patients)...")
at_df = compute_at_hourly(hourly_full, weights, pop_stats, multipliers, CORE_VITALS)

# Per-patient summary
at_summary = at_df.groupby("stay_id")["at_value"].agg(
    at_max="max", at_mean="mean", at_std="std", at_last="last"
).reset_index()
at_summary["at_std"] = at_summary["at_std"].fillna(0)

# Save
at_df.to_csv(os.path.join(PROCESSED_DIR, "at_hourly.csv.gz"), index=False, compression="gzip")
at_summary.to_csv(os.path.join(PROCESSED_DIR, "at_summary.csv"), index=False)

calibration = {
    "weights": weights,
    "multipliers": multipliers,
    "pop_stats": {k: {"mean": v["mean"], "std": v["std"]} for k, v in pop_stats.items()},
    "computed_on": "training_set_only",
    "n_train": len(train_idx),
}
with open(os.path.join(PROCESSED_DIR, "at_calibration.json"), "w") as f:
    json.dump(calibration, f, indent=2)

print(f"\nA(t) max per patient: {at_summary['at_max'].mean():.3f} +/- {at_summary['at_max'].std():.3f}")
print(f"A(t) mean: {at_summary['at_mean'].mean():.3f} +/- {at_summary['at_mean'].std():.3f}")

In [ ]:
# Merge A(t) into features (drop any old A(t) cols first to avoid duplicates)
features = features.drop(columns=[c for c in ['at_max', 'at_mean', 'at_std', 'at_last'] if c in features.columns])
features = features.merge(at_summary, on="stay_id", how="left")

# A(t) distribution plot (uses train+val+test data, but A(t) values are train-calibrated)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ["at_max", "at_mean"],
                           ["max A(t) per patient", "mean A(t) per patient"]):
    for label, color in [(0, "steelblue"), (1, "firebrick")]:
        subset = features[features["in_hospital_mortality"] == label][col]
        ax.hist(subset, bins=50, alpha=0.6, color=color,
                label=f"{'Survived' if label==0 else 'Died'}")
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
    ax.legend()
plt.suptitle("A(t) Distribution by Outcome (train-calibrated)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "at_distribution.png"), dpi=200)
plt.show()

# Save updated features
features.to_csv(os.path.join(PROCESSED_DIR, "features_xgboost.csv"), index=False)
print(f"Updated features saved with train-calibrated A(t).")

---
## Step 7: XGBoost Experiments — All Conditions

Four conditions: Full, NEWS2-set, CAS-4, CAS-4 hybrid A

In [ ]:
from xgboost import XGBClassifier

# Define feature sets
cas4_cols = [f"{v}{s}" for v in CORE_VITALS for s in SUFFIXES if f"{v}{s}" in features.columns]
news2_vitals = CORE_VITALS + ["sbp", "gcs_total"]
news2_cols = [f"{v}{s}" for v in news2_vitals for s in SUFFIXES if f"{v}{s}" in features.columns]
full_cols = [c for c in features.columns if c not in
             ["stay_id", "in_hospital_mortality", "gender", "at_max", "at_mean", "at_std", "at_last"]]
at_feature_cols = ["at_max", "at_mean", "at_std", "at_last"]
cas4_hybrid_cols = cas4_cols + [c for c in at_feature_cols if c in features.columns]

FEATURE_SETS = {
    "Full": full_cols,
    "NEWS2-set": news2_cols,
    "CAS-4": cas4_cols,
    "CAS-4 hybrid A": cas4_hybrid_cols,
}

for name, cols in FEATURE_SETS.items():
    print(f"  {name}: {len(cols)} features")

In [ ]:
# Helper functions
def compute_metrics(y_true, y_prob, name):
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_prob)
    fpr80 = fpr_arr[np.argmin(np.abs(tpr_arr - 0.80))]
    fpr90 = fpr_arr[np.argmin(np.abs(tpr_arr - 0.90))]
    return {"name": name, "auroc": auroc, "auprc": auprc, "brier": brier,
            "fpr_at_sens80": fpr80, "fpr_at_sens90": fpr90}

def bootstrap_ci(y_true, y_prob, n_boot=1000):
    scores = []
    n = len(y_true)
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        try:
            scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
        except ValueError:
            continue
    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

scale_pw = (y.loc[train_idx] == 0).sum() / max((y.loc[train_idx] == 1).sum(), 1)
print(f"scale_pos_weight: {scale_pw:.2f}")

In [ ]:
# Run XGBoost for each condition
all_results = []
all_predictions = {}        # test-set predicted probabilities
all_val_predictions = {}    # validation-set predicted probabilities (used by Step 13b Platt scaling)
roc_data = {}
xgb_models = {}

for cond_name, cols in FEATURE_SETS.items():
    avail = [c for c in cols if c in features.columns]
    scaler = StandardScaler()

    X_tr = scaler.fit_transform(features.loc[train_idx, avail])
    X_va = scaler.transform(features.loc[val_idx, avail])
    X_te = scaler.transform(features.loc[test_idx, avail])
    y_tr = y.loc[train_idx].values
    y_va = y.loc[val_idx].values
    y_te = y.loc[test_idx].values

    model = XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, random_state=RANDOM_STATE,
        eval_metric="aucpr", early_stopping_rounds=30, verbosity=0,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    y_prob = model.predict_proba(X_te)[:, 1]
    metrics = compute_metrics(y_te, y_prob, f"{cond_name} + XGBoost")
    lo, hi = bootstrap_ci(y_te, y_prob)
    metrics["ci_low"] = lo
    metrics["ci_high"] = hi

    all_results.append(metrics)
    roc_data[metrics["name"]] = (y_te, y_prob)
    all_predictions[metrics["name"]] = y_prob
    # also save validation-set predictions for downstream recalibration analyses (Step 13b)
    all_val_predictions[f"{cond_name} + XGBoost"] = model.predict_proba(X_va)[:, 1]
    xgb_models[cond_name] = (model, scaler, avail)

    print(f"  {cond_name}: AUROC={metrics['auroc']:.4f} ({lo:.3f}-{hi:.3f}), "
          f"AUPRC={metrics['auprc']:.4f}")

---
## Step 8: NEWS2 Scoring Baseline

In [ ]:
def news2_score_row(row):
    score = 0
    rr = row.get("resp_rate_last", row.get("resp_rate_mean", np.nan))
    if pd.notna(rr):
        if rr <= 8: score += 3
        elif rr <= 11: score += 1
        elif rr <= 20: score += 0
        elif rr <= 24: score += 2
        else: score += 3
    spo2 = row.get("spo2_last", row.get("spo2_mean", np.nan))
    if pd.notna(spo2):
        if spo2 <= 91: score += 3
        elif spo2 <= 93: score += 2
        elif spo2 <= 95: score += 1
        else: score += 0
    sbp = row.get("sbp_last", row.get("sbp_mean", np.nan))
    if pd.notna(sbp):
        if sbp <= 90: score += 3
        elif sbp <= 100: score += 2
        elif sbp <= 110: score += 1
        elif sbp <= 219: score += 0
        else: score += 3
    hr = row.get("heart_rate_last", row.get("heart_rate_mean", np.nan))
    if pd.notna(hr):
        if hr <= 40: score += 3
        elif hr <= 50: score += 1
        elif hr <= 90: score += 0
        elif hr <= 110: score += 1
        elif hr <= 130: score += 2
        else: score += 3
    temp = row.get("temperature_last", row.get("temperature_mean", np.nan))
    if pd.notna(temp):
        if temp <= 35.0: score += 3
        elif temp <= 36.0: score += 1
        elif temp <= 38.0: score += 0
        elif temp <= 39.0: score += 1
        else: score += 2
    return score

features["news2_total"] = features.apply(news2_score_row, axis=1)

y_te = y.loc[test_idx].values
news2_scores = features.loc[test_idx, "news2_total"].values
metrics_news2 = compute_metrics(y_te, news2_scores / 20.0, "NEWS2 scoring")
lo, hi = bootstrap_ci(y_te, news2_scores / 20.0)
metrics_news2["ci_low"] = lo
metrics_news2["ci_high"] = hi
all_results.append(metrics_news2)
roc_data["NEWS2 scoring"] = (y_te, news2_scores / 20.0)
all_predictions["NEWS2 scoring"] = news2_scores / 20.0

print(f"NEWS2: AUROC={metrics_news2['auroc']:.4f} ({lo:.3f}-{hi:.3f})")

---
## Step 9: Feature Reduction Curve — CENTRAL ANALYSIS

Systematically evaluate 9 sensor configurations from full hospital monitoring
to a single pulse sensor. This is the **key new contribution** of the paper.

| Level | Vitals | Sensors | Cost |
|-------|--------|---------|------|
| L1-Full | 50+ features | Hospital ICU | $$$$ |
| L2-AllVitals | All vitals + demo | Bedside | $$$ |
| L3-NEWS2 | HR,RR,SpO2,Temp,SBP,GCS | Nurse + cuff | $$ |
| L4-5vitals | HR,RR,SpO2,Temp,SBP | Wearable + cuff | ~$50 |
| **L5-CAS4** | **HR,RR,SpO2,Temp** | **MAX30102+radar+IR** | **~$30** |
| L5b-CAS4+At | HR,RR,SpO2,Temp + A(t) | Same + domain knowledge | ~$30 |
| L6-3vitals | HR,SpO2,RR | MAX30102 + radar | ~$20 |
| L7-2vitals | HR,SpO2 | MAX30102 only | ~$5 |
| L8-1vital | HR only | Pulse sensor | ~$2 |

In [ ]:
import xgboost as xgb

REDUCTION_LEVELS = [
    {'name': 'L1-Full',       'vitals': list(VITAL_SIGN_ITEMS.keys()), 'include_demo': True,  'add_at': False, 'use_all': True,  'cost': '$$$$'},
    {'name': 'L2-AllVitals',  'vitals': list(VITAL_SIGN_ITEMS.keys()), 'include_demo': True,  'add_at': False, 'use_all': False, 'cost': '$$$'},
    {'name': 'L3-NEWS2',      'vitals': ['heart_rate','resp_rate','spo2','temperature','sbp','gcs_total'], 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '$$'},
    {'name': 'L4-5vitals',    'vitals': ['heart_rate','resp_rate','spo2','temperature','sbp'], 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '~$50'},
    {'name': 'L5-CAS4',       'vitals': CORE_VITALS, 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '~$30'},
    {'name': 'L5b-CAS4+At',   'vitals': CORE_VITALS, 'include_demo': False, 'add_at': True,  'use_all': False, 'cost': '~$30'},
    {'name': 'L6-3vitals',    'vitals': ['heart_rate','spo2','resp_rate'], 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '~$20'},
    {'name': 'L7-2vitals',    'vitals': ['heart_rate','spo2'], 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '~$5'},
    {'name': 'L8-1vital',     'vitals': ['heart_rate'], 'include_demo': False, 'add_at': False, 'use_all': False, 'cost': '~$2'},
]

reduction_results = []
reduction_features = {}
reduction_predictions = {}

print(f'Testing {len(REDUCTION_LEVELS)} sensor reduction levels')
print('=' * 80)

for level in REDUCTION_LEVELS:
    name = level['name']

    if level['use_all']:
        exclude = {'stay_id', target, 'gender', 'icu_type', 'age_group',
                   'at_max', 'at_mean', 'at_std', 'at_last'}
        cols = [c for c in features.select_dtypes(include=[np.number]).columns if c not in exclude]
    else:
        cols = []
        for v in level['vitals']:
            for s in SUFFIXES:
                col = f'{v}{s}'
                if col in features.columns:
                    cols.append(col)

    if level['include_demo'] and 'age' in features.columns:
        cols.append('age')

    if level['add_at']:
        for c in ['at_max', 'at_mean', 'at_std', 'at_last']:
            if c in features.columns and c not in cols:
                cols.append(c)

    avail = [c for c in cols if c in features.columns]
    reduction_features[name] = list(avail)
    if len(avail) == 0:
        print(f'  {name}: NO features, skipping')
        continue

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(features.loc[train_idx, avail])
    X_va = scaler.transform(features.loc[val_idx, avail])
    X_te = scaler.transform(features.loc[test_idx, avail])
    y_tr = y.loc[train_idx].values
    y_va = y.loc[val_idx].values

    model = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw, random_state=RANDOM_STATE,
        eval_metric='aucpr', early_stopping_rounds=30, verbosity=0)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    y_prob = model.predict_proba(X_te)[:, 1]

    auroc = roc_auc_score(y_te, y_prob)
    lo, hi = bootstrap_ci(y_te, y_prob)
    auprc = average_precision_score(y_te, y_prob)
    brier = brier_score_loss(y_te, y_prob)

    eps = 1e-7
    p_clip = np.clip(y_prob, eps, 1-eps)
    logit_p = np.log(p_clip / (1-p_clip))
    lr_c = LogisticRegression(fit_intercept=True, max_iter=1000)
    lr_c.fit(logit_p.reshape(-1,1), y_te)
    cal_sl = float(lr_c.coef_[0][0])

    fpr_arr, tpr_arr, _ = roc_curve(y_te, y_prob)
    fpr80 = float(fpr_arr[np.argmin(np.abs(tpr_arr - 0.80))])

    n = len(y_te); pt = 0.10
    y_pred_dca = (y_prob >= pt).astype(int)
    tp = ((y_pred_dca==1) & (y_te==1)).sum()
    fp = ((y_pred_dca==1) & (y_te==0)).sum()
    nb_010 = (tp/n) - (fp/n) * (pt/(1-pt))

    full_auroc = reduction_results[0]['auroc'] if reduction_results else auroc
    pct = auroc / full_auroc * 100 if full_auroc > 0 else 100

    reduction_results.append({
        'level': name, 'n_features': len(avail), 'cost': level['cost'],
        'auroc': round(auroc,4), 'ci_low': round(lo,4), 'ci_high': round(hi,4),
        'auroc_ci': f'{lo:.3f}-{hi:.3f}',
        'auprc': round(auprc,4), 'brier': round(brier,4),
        'cal_slope': round(cal_sl,3), 'fpr_at_sens80': round(fpr80,3),
        'nb_at_pt10': round(nb_010,4), 'pct_of_full': round(pct,1),
    })
    reduction_predictions[name] = y_prob
    all_predictions[f'{name} + XGBoost'] = y_prob

    print(f'  {name:15s} | {len(avail):3d} feat | AUROC={auroc:.4f} ({lo:.3f}-{hi:.3f}) |'
          f' Brier={brier:.4f} | Slope={cal_sl:.3f} | {pct:.1f}% of full')

red_df = pd.DataFrame(reduction_results)
red_df.to_csv(os.path.join(RESULTS_DIR, 'feature_reduction_curve.csv'), index=False)
print(f'\nSaved: {os.path.join(RESULTS_DIR, "feature_reduction_curve.csv")}')

In [ ]:
# Central figure: feature reduction degradation curve
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
x_labels = [r['level'].split('-',1)[1] if '-' in r['level'] else r['level'] for r in reduction_results]
x = range(len(red_df))

# Panel A: AUROC
ax = axes[0, 0]
ax.plot(x, red_df['auroc'], 'o-', color='#2E86C1', linewidth=2.5, markersize=8)
ax.fill_between(x, red_df['ci_low'], red_df['ci_high'], alpha=0.15, color='#2E86C1')
ax.set_ylabel('AUROC', fontsize=12)
ax.set_title('A. Discrimination (AUROC)', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.grid(alpha=0.3); ax.set_ylim([0.5, max(red_df['auroc'])*1.08])
for idx_r, row in red_df.iterrows():
    if row['level'] == 'L5-CAS4':
        ax.annotate(f'CAS-4\nAUROC={row["auroc"]:.3f}',
            xy=(idx_r, row['auroc']), xytext=(idx_r+0.8, row['auroc']+0.015),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red', fontweight='bold')

# Panel B: Calibration Slope
ax = axes[0, 1]
ax.plot(x, red_df['cal_slope'], 's-', color='#E74C3C', linewidth=2.5, markersize=8)
ax.axhline(y=1.0, color='green', linestyle='--', linewidth=1, label='Perfect')
ax.set_ylabel('Calibration Slope', fontsize=12)
ax.set_title('B. Calibration Slope', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.grid(alpha=0.3); ax.legend(fontsize=10)

# Panel C: Brier Score
ax = axes[1, 0]
ax.plot(x, red_df['brier'], 'D-', color='#27AE60', linewidth=2.5, markersize=8)
ax.set_ylabel('Brier Score (lower=better)', fontsize=12)
ax.set_title('C. Overall Performance (Brier)', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.grid(alpha=0.3)

# Panel D: % of Full
ax = axes[1, 1]
bars = ax.bar(x, red_df['pct_of_full'], color='#F39C12', alpha=0.8, edgecolor='#D68910')
ax.axhline(y=95, color='green', linestyle='--', linewidth=1, label='95% threshold')
ax.axhline(y=90, color='orange', linestyle='--', linewidth=1, label='90% threshold')
ax.set_ylabel('% of Full Performance', fontsize=12)
ax.set_title('D. Relative Performance', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
ax.grid(alpha=0.3, axis='y'); ax.legend(fontsize=10)
for bar, pct in zip(bars, red_df['pct_of_full']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{pct:.1f}%',
           ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Feature Reduction Curve: From Hospital to Wearable',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'fig_CENTRAL_reduction_curve.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('\n' + '='*70)
print('KEY FINDING: Feature Reduction Analysis')
print('='*70)
for _, row in red_df.iterrows():
    marker = ' <<< CAS-4' if row['level']=='L5-CAS4' else ''
    if row['level']=='L5b-CAS4+At': marker = ' <<< CAS-4+A(t)'
    print(f'  {row["level"]:15s}: AUROC={row["auroc"]:.4f} ({row["pct_of_full"]:5.1f}% of full) | '
          f'Slope={row["cal_slope"]:.3f} | Cost={row["cost"]:6s}{marker}')

drops = []
for i in range(1, len(red_df)):
    drop = red_df.iloc[i-1]['auroc'] - red_df.iloc[i]['auroc']
    drops.append((red_df.iloc[i]['level'], round(drop, 4)))
drops.sort(key=lambda x: x[1], reverse=True)
print(f'\nKnee point (largest drop): {drops[0][0]} (delta={drops[0][1]})')

---
## Step 9b: L3-NEWS2 vs L4-5vitals Identity Diagnostic (Reviewer IV.12)

**Reviewer IV.12:** In Table II, rows L3-NEWS2 (45 features, includes GCS) and L4-5vitals (45 features, no GCS) report **identical AUROC, calibration slope, Brier score, and FPR@80 to four decimals**. This is suspicious. The most likely explanation is that GCS in MIMIC-IV ICU patients is uninformative (saturated at 15 for sedated/intubated patients, or missing → forward-filled to a constant), making its inclusion equivalent to its exclusion at the model level.

This cell verifies the explanation empirically.


In [ ]:
# Step 9b: Diagnose why L3-NEWS2 and L4-5vitals produce identical metrics.
# Hypothesis: GCS is uninformative in this MIMIC-IV ICU cohort due to saturation
# and missing-data treatment. If true, we have an additional finding worth
# reporting (NEWS2 GCS component contributes nothing on this task).

# Identify GCS column(s) in the feature matrix
gcs_cols = [c for c in features.columns if 'gcs' in c.lower() or 'glasgow' in c.lower()
            or 'avpu' in c.lower() or 'consciousness' in c.lower()]
print(f'GCS-related feature columns: {gcs_cols}')

if not gcs_cols:
    print('No GCS column found — likely never extracted into features matrix.')
    print('This itself explains L3=L4: GCS was never in the model.')
else:
    for col in gcs_cols:
        s = features[col]
        print(f'\n  Column "{col}":')
        print(f'    n total:     {len(s)}')
        print(f'    n null:      {s.isna().sum()} ({s.isna().mean()*100:.1f}%)')
        print(f'    n unique:    {s.nunique()}')
        if s.nunique() > 0:
            print(f'    range:       [{s.min()}, {s.max()}]')
            print(f'    median:      {s.median()}')
            top = s.value_counts(dropna=False).head(5)
            print(f'    top values:  {dict(top)}')

# Check whether the trained models for L3 and L4 used the same column set
# Reconstruct reduction_features if it doesn't exist yet (e.g., the
# Step 9 loop was run before this dict was being populated).
if 'reduction_features' not in dir():
    reduction_features = {}
    for _level in REDUCTION_LEVELS:
        if _level['use_all']:
            _exclude = {'stay_id', target, 'gender', 'icu_type', 'age_group',
                        'at_max', 'at_mean', 'at_std', 'at_last'}
            _cols = [c for c in features.select_dtypes(include=[np.number]).columns
                     if c not in _exclude]
        else:
            _cols = []
            for _v in _level['vitals']:
                for _s in SUFFIXES:
                    _col = f'{_v}{_s}'
                    if _col in features.columns:
                        _cols.append(_col)
        if _level['include_demo'] and 'age' in features.columns:
            _cols.append('age')
        if _level['add_at']:
            for _c in ['at_max', 'at_mean', 'at_std', 'at_last']:
                if _c in features.columns and _c not in _cols:
                    _cols.append(_c)
        reduction_features[_level['name']] = [c for c in _cols if c in features.columns]

l3_cols = sorted(reduction_features.get('L3-NEWS2', []))
l4_cols = sorted(reduction_features.get('L4-5vitals', []))
print(f'\nL3 column count: {len(l3_cols)}')
print(f'L4 column count: {len(l4_cols)}')
shared = set(l3_cols) & set(l4_cols)
l3_only = set(l3_cols) - set(l4_cols)
l4_only = set(l4_cols) - set(l3_cols)
print(f'Shared:    {len(shared)}')
print(f'L3 only:   {sorted(l3_only)[:20]}')
print(f'L4 only:   {sorted(l4_only)[:20]}')

# If L3 and L4 column sets are literally identical, that explains the identical
# metrics directly (the model is the same).
if l3_only == set() and l4_only == set():
    print('\n→ L3 and L4 used the IDENTICAL feature columns. Identical metrics confirmed.')
    print('  This is a manuscript finding to report explicitly: under the present')
    print('  feature-engineering scheme, GCS was not added to L3 (or was dropped),')
    print('  so L3-NEWS2 and L4-5vitals are the same model. The NEWS2 distinction')
    print('  in the paper should either restore GCS as a feature (recommended) or')
    print('  collapse the two rows in Table II.')


---
## Step 10: Additional Models — CatBoost & Random Forest

CatBoost and Random Forest applied to CAS-4 and CAS-4 hybrid configurations.

In [ ]:
# Install catboost if needed
try:
    from catboost import CatBoostClassifier
except ImportError:
    print("Installing catboost...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostClassifier

from sklearn.ensemble import RandomForestClassifier

NEW_MODELS = {
    'CatBoost': lambda: CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05,
        auto_class_weights='Balanced', random_state=RANDOM_STATE, verbose=0),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=500, max_depth=10, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1),
}

for cond_name in ['CAS-4', 'CAS-4 hybrid A']:
    if cond_name not in FEATURE_SETS:
        continue
    cols = FEATURE_SETS[cond_name]
    avail = [c for c in cols if c in features.columns]

    for model_name, model_fn in NEW_MODELS.items():
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(features.loc[train_idx, avail])
        X_te = scaler.transform(features.loc[test_idx, avail])
        y_tr = y.loc[train_idx].values
        y_te_local = y.loc[test_idx].values

        m = model_fn()
        m.fit(X_tr, y_tr)
        y_prob = m.predict_proba(X_te)[:, 1]

        auroc = roc_auc_score(y_te_local, y_prob)
        lo, hi = bootstrap_ci(y_te_local, y_prob)
        auprc = average_precision_score(y_te_local, y_prob)
        brier = brier_score_loss(y_te_local, y_prob)
        fpr_arr, tpr_arr, _ = roc_curve(y_te_local, y_prob)
        fpr80 = float(fpr_arr[np.argmin(np.abs(tpr_arr - 0.80))])

        key = f'{cond_name} + {model_name}'
        all_results.append({
            'name': key, 'auroc': auroc, 'auprc': auprc, 'brier': brier,
            'ci_low': lo, 'ci_high': hi, 'fpr_at_sens80': fpr80, 'fpr_at_sens90': np.nan
        })
        all_predictions[key] = y_prob
        roc_data[key] = (y_te_local, y_prob)
        xgb_models[f'{cond_name}_{model_name}'] = (m, scaler, avail)

        print(f'  {key}: AUROC={auroc:.4f} ({lo:.3f}-{hi:.3f})')

results_df = pd.DataFrame(all_results).sort_values('auroc', ascending=False)
results_df.to_csv(os.path.join(RESULTS_DIR, 'all_model_results.csv'), index=False)
print(f'\nTotal model/condition combos: {len(all_results)}')

---
## Step 11: SHAP Feature Importance

For CAS-4 hybrid A + XGBoost. Exports both summary plot and explicit
ranking CSV (mean absolute SHAP value per feature) for use in the paper.

In [ ]:
import shap

cas4h_model, cas4h_scaler, cas4h_cols = xgb_models['CAS-4 hybrid A']
X_te_shap = cas4h_scaler.transform(features.loc[test_idx, cas4h_cols])

explainer = shap.TreeExplainer(cas4h_model)
shap_values = explainer.shap_values(X_te_shap)

# v4: Export explicit ranking CSV alongside summary plot
shap_importance = pd.DataFrame({
    'feature': cas4h_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance['rank'] = shap_importance.index + 1
shap_importance.to_csv(os.path.join(RESULTS_DIR, 'shap_ranking.csv'), index=False)

print("Top 15 SHAP features (CAS-4 hybrid A + XGBoost):")
print(shap_importance.head(15).to_string(index=False))

# Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_te_shap, feature_names=cas4h_cols,
                  max_display=15, show=False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_shap_cas4_hybrid.png'),
            dpi=300, bbox_inches='tight')
plt.show()

### Step 11b — SHAP feature importance on L3-NEWS2 (with GCS)


In [ ]:
# SHAP feature importance on the L3-NEWS2 model.
# Retrains XGBoost on the L3-NEWS2 feature set (CORE_VITALS + sbp + gcs_total)
# so SHAP values reflect the model that includes GCS components. Self-contained:
# uses reduction_features (populated in Step 9, Cell 36) and the existing split.
import shap

l3_cols = reduction_features['L3-NEWS2']
print(f'L3-NEWS2 feature set: {len(l3_cols)} columns')
print(f'  GCS-related: {[c for c in l3_cols if "gcs" in c.lower()]}')

scaler_l3 = StandardScaler()
X_tr_l3 = scaler_l3.fit_transform(features.loc[train_idx, l3_cols])
X_va_l3 = scaler_l3.transform(features.loc[val_idx,   l3_cols])
X_te_l3 = scaler_l3.transform(features.loc[test_idx,  l3_cols])
y_tr = y.loc[train_idx].values
y_va = y.loc[val_idx].values
scale_pw_l3 = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

model_l3 = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pw_l3, random_state=RANDOM_STATE,
    eval_metric='aucpr', early_stopping_rounds=30, verbosity=0,
)
model_l3.fit(X_tr_l3, y_tr, eval_set=[(X_va_l3, y_va)], verbose=False)

explainer_l3 = shap.TreeExplainer(model_l3)
shap_values_l3 = explainer_l3.shap_values(X_te_l3)

shap_importance_l3 = pd.DataFrame({
    'feature': l3_cols,
    'mean_abs_shap': np.abs(shap_values_l3).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
shap_importance_l3['rank'] = shap_importance_l3.index + 1
shap_importance_l3.to_csv(os.path.join(RESULTS_DIR, 'shap_ranking_l3_news2.csv'), index=False)

print('\nTop 15 SHAP features (L3-NEWS2 + XGBoost):')
print(shap_importance_l3.head(15).to_string(index=False))

# Highlight the GCS contribution
gcs_in_top15 = shap_importance_l3.head(15)['feature'].str.contains('gcs').sum()
gcs_rank = shap_importance_l3[
    shap_importance_l3['feature'].str.startswith('gcs_total_')
].head(3)
print(f'\nGCS features in top 15: {gcs_in_top15}')
if not gcs_rank.empty:
    print('Highest-ranked gcs_total features:')
    print(gcs_rank.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_l3, X_te_l3, feature_names=l3_cols,
                  max_display=15, show=False)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_shap_l3_news2.png'),
            dpi=300, bbox_inches='tight')
plt.show()


---
## Step 12: Calibration Analysis

Following Van Calster et al. (2019). Metrics: CITL, calibration slope, Brier, Hosmer-Lemeshow.

In [ ]:
from sklearn.calibration import calibration_curve

def cal_slope(y_true, y_prob):
    eps = 1e-7
    p = np.clip(y_prob, eps, 1-eps)
    logit_p = np.log(p/(1-p))
    lr = LogisticRegression(fit_intercept=True, max_iter=1000)
    lr.fit(logit_p.reshape(-1,1), y_true)
    return float(lr.coef_[0][0])

def cal_itl(y_true, y_prob):
    return float(y_prob.mean() - y_true.mean())

def hosmer_lemeshow(y_true, y_prob, g=10):
    df_hl = pd.DataFrame({'y': y_true, 'p': y_prob})
    df_hl['grp'] = pd.qcut(df_hl['p'], g, duplicates='drop')
    obs = df_hl.groupby('grp')['y'].sum()
    exp = df_hl.groupby('grp')['p'].sum()
    n_grp = df_hl.groupby('grp')['y'].count()
    hl = (((obs-exp)**2) / (exp*(1-exp/n_grp)+1e-10)).sum()
    pval = 1 - stats.chi2.cdf(hl, len(obs)-2)
    return float(hl), float(pval)

y_te = y.loc[test_idx].values
cal_results = []

for key, yp in all_predictions.items():
    slope = cal_slope(y_te, yp)
    citl = cal_itl(y_te, yp)
    hl_stat, hl_p = hosmer_lemeshow(y_te, yp)
    brier = brier_score_loss(y_te, yp)
    prev = y_te.mean()
    sc_brier = 1 - brier/(prev*(1-prev)) if prev*(1-prev)>0 else 0
    cal_results.append({'model': key, 'cal_slope': slope, 'citl': citl,
        'brier': brier, 'scaled_brier': sc_brier, 'hl_stat': hl_stat, 'hl_p': hl_p})
    print(f'  {key}: slope={slope:.3f}, CITL={citl:.4f}, Brier={brier:.4f}')

cal_df = pd.DataFrame(cal_results)
cal_df.to_csv(os.path.join(RESULTS_DIR, 'calibration_results.csv'), index=False)

# Calibration plots
plot_keys = [k for k in all_predictions if 'CAS-4 hybrid' in k or 'Full + XGB' in k or 'NEWS2' in k][:6]
colors_cal = ['#2E86C1','#E74C3C','#27AE60','#F39C12','#8E44AD','#1ABC9C']
fig, ax = plt.subplots(figsize=(8,8))
for i, key in enumerate(plot_keys):
    frac, mp = calibration_curve(y_te, all_predictions[key], n_bins=10, strategy='quantile')
    ax.plot(mp, frac, 's-', color=colors_cal[i%6], linewidth=2, label=key)
ax.plot([0,1],[0,1],'k--',linewidth=1,label='Perfect')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed proportion')
ax.set_title('Calibration — MIMIC-IV Test Set')
ax.legend(fontsize=8, loc='lower right'); ax.set_xlim([0,1]); ax.set_ylim([0,1]); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'fig_calibration.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Calibration analysis complete.')

---
## Step 12b: Decile-Level HL Diagnostic (Supplementary)

**Reviewer II.4:** The Hosmer-Lemeshow test rejects calibration for all models due to large sample size (n≈7,798 makes HL highly sensitive to small deviations). To verify that calibration is clinically acceptable in the high-risk decile (where clinical decisions are made), we report decile-level observed vs. expected events for the top-3 recommended models.

Output: `results/calibration_decile_diagnostic.csv` (for Supplementary Table)


In [ ]:
# Step 12b: Decile-level observed/expected diagnostic for top recommended models.
# This addresses the HL-test problem: HL p<0.001 for all models can mislead
# at large n. Decile-level inspection lets reviewers see WHERE miscalibration
# (if any) lives — particularly in the high-risk decile where decisions are made.

def decile_diagnostic(y_true, y_prob, label, n_bins=10):
    """Return per-decile counts: n, observed events, expected events, ratio."""
    df = pd.DataFrame({'y': y_true, 'p': y_prob})
    # qcut with duplicates='drop' to handle ties at decile boundaries
    df['decile'] = pd.qcut(df['p'], n_bins, labels=False, duplicates='drop')
    rows = []
    for d in sorted(df['decile'].dropna().unique()):
        sub = df[df['decile'] == d]
        observed = int(sub['y'].sum())
        expected = float(sub['p'].sum())
        n = len(sub)
        rows.append({
            'model': label,
            'decile': int(d) + 1,
            'n': n,
            'p_min': round(float(sub['p'].min()), 4),
            'p_max': round(float(sub['p'].max()), 4),
            'p_mean': round(float(sub['p'].mean()), 4),
            'observed': observed,
            'expected': round(expected, 2),
            'observed_rate': round(observed / n, 4) if n else 0.0,
            'expected_rate': round(expected / n, 4) if n else 0.0,
            'obs_exp_ratio': round(observed / expected, 3) if expected > 0 else None,
        })
    return rows

# Top-3 models we want to scrutinize for decile-level fit
top_models_keys = [
    ('Full + XGBoost',         'L1-Full + XGBoost'),
    ('CAS-4 hybrid A + XGBoost', 'CAS-4 hybrid A + XGBoost (recommended)'),
    ('CAS-4 + CatBoost',         'CAS-4 + CatBoost'),
]

decile_rows = []
for src_key, label in top_models_keys:
    if src_key in all_predictions:
        decile_rows.extend(decile_diagnostic(y_te, all_predictions[src_key], label))
    else:
        print(f'  ⚠ {src_key} not in all_predictions; skipping')

decile_df = pd.DataFrame(decile_rows)
print('Decile-level diagnostic (head):')
print(decile_df.head(20).to_string(index=False))

# Highlight high-risk decile fit specifically (the one that matters clinically)
print('\n--- High-risk decile (decile 10) summary ---')
hr_decile = decile_df[decile_df['decile'] == decile_df['decile'].max()]
for _, row in hr_decile.iterrows():
    print(f"  {row['model']:50s}: n={row['n']}, obs={row['observed']}, "
          f"exp={row['expected']:.1f}, ratio={row['obs_exp_ratio']}")

out_path = os.path.join(RESULTS_DIR, 'calibration_decile_diagnostic.csv')
decile_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')


---
## Step 12c: L7 Calibration Sanity Check (Reviewer II.3)

**Reviewer II.3:** A calibration slope of 2.654 at L7 (HR + SpO₂ only) is unusual — slope > 2 typically indicates that the model's predicted-probability distribution is severely compressed near the base rate, which is more characteristic of underfitting than overconfidence. Since L8 (HR alone) has slope=1.457 < L7=2.654 but lower AUROC (0.635 vs 0.684), this non-monotonicity warrants direct visual inspection.

This cell:
1. Plots predicted-probability histogram for L6, L7, L8 side by side
2. Reproduces the slope estimate via two methods (logistic recalibration vs. weighted least squares on logit) — divergence indicates numerical instability
3. Bootstraps the slope to report a 95% CI


In [ ]:
# Step 12c: Sanity-check the L7 calibration slope of 2.654.
# Goals: (a) visualize the predicted-probability distribution shape,
#        (b) recompute slope by an independent method,
#        (c) bootstrap the slope estimate to expose numerical instability.

import matplotlib.pyplot as plt

required_keys = ['L6-3vitals', 'L7-2vitals', 'L8-1vital']
have_all = all(k in reduction_predictions for k in required_keys)
if not have_all:
    print(f'⚠ Missing reduction predictions for L6/L7/L8; ensure Step 9 ran. Found keys: {sorted(reduction_predictions.keys())}')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, key in zip(axes, required_keys):
        p = reduction_predictions[key]
        ax.hist(p, bins=50, alpha=0.7, edgecolor='black')
        ax.axvline(y_te.mean(), color='red', linestyle='--', label=f'Base rate={y_te.mean():.3f}')
        ax.set_title(f'{key}: P(deterioration | x)')
        ax.set_xlabel('Predicted probability')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('Count')
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig_l7_calibration_diagnostic.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

    # Summary statistics of the predicted-probability distribution
    print('Predicted-probability distribution summary:')
    print(f'{"Level":12s} {"min":>8s} {"q25":>8s} {"median":>8s} {"q75":>8s} {"max":>8s} {"std":>8s}')
    for key in required_keys:
        p = reduction_predictions[key]
        q = np.percentile(p, [25, 50, 75])
        print(f'{key:12s} {p.min():>8.4f} {q[0]:>8.4f} {q[1]:>8.4f} {q[2]:>8.4f} {p.max():>8.4f} {p.std():>8.4f}')

    # Bootstrap slope estimate (n=1000) for L7 specifically
    print('\nBootstrap calibration slope for L7 (n=1000, seed=42):')
    rng_l7 = np.random.RandomState(42)
    p_l7 = reduction_predictions['L7-2vitals']
    slopes = []
    for _ in range(1000):
        idx = rng_l7.choice(len(y_te), len(y_te), replace=True)
        if len(np.unique(y_te[idx])) < 2:
            continue
        try:
            slopes.append(cal_slope(y_te[idx], p_l7[idx]))
        except Exception:
            continue
    slopes = np.array(slopes)
    print(f'  Point estimate: {cal_slope(y_te, p_l7):.4f}')
    print(f'  Bootstrap median: {np.median(slopes):.4f}')
    print(f'  Bootstrap 95% CI: [{np.percentile(slopes, 2.5):.4f}, {np.percentile(slopes, 97.5):.4f}]')
    print(f'  Bootstrap std: {slopes.std():.4f}')
    if slopes.std() > 0.5:
        print('  ⚠ Large bootstrap std — slope estimate is numerically unstable; consider')
        print('    reporting the median + CI rather than the point estimate in the paper.')


---
## Step 13: Decision Curve Analysis

Following Vickers et al. (2006). Reports explicit net benefit at p_t=0.10
for each model (used in the paper text).

In [ ]:
def net_benefit(y_true, y_prob, pt):
    n = len(y_true)
    if pt >= 1 or pt <= 0: return 0.0
    yp = (y_prob >= pt).astype(int)
    tp = ((yp==1)&(y_true==1)).sum(); fp = ((yp==1)&(y_true==0)).sum()
    return (tp/n) - (fp/n)*(pt/(1-pt))

thresholds = np.arange(0.01, 0.51, 0.01)
prevalence = y_te.mean()
dca_records = []
for pt in thresholds:
    dca_records.append({'threshold':pt, 'model':'Treat All',
                        'net_benefit': prevalence-(1-prevalence)*(pt/(1-pt))})
    dca_records.append({'threshold':pt, 'model':'Treat None', 'net_benefit': 0.0})

dca_keys = [k for k in all_predictions if 'CAS-4 hybrid' in k or 'Full + XGB' in k or 'NEWS2' in k][:6]
for key in dca_keys:
    for pt in thresholds:
        dca_records.append({'threshold':pt, 'model':key,
                            'net_benefit': net_benefit(y_te, all_predictions[key], pt)})

dca_df = pd.DataFrame(dca_records)
dca_df.to_csv(os.path.join(RESULTS_DIR, 'dca_results.csv'), index=False)

# v4: Explicit NB at p_t=0.10 for each model (used in paper text)
print("\n" + "="*60)
print("Net Benefit at threshold probability p_t = 0.10")
print("="*60)
nb_010_records = []
for key in dca_keys + ['Treat All']:
    if key == 'Treat All':
        nb = prevalence - (1-prevalence)*(0.10/(1-0.10))
    else:
        nb = net_benefit(y_te, all_predictions[key], 0.10)
    nb_010_records.append({'model': key, 'nb_at_pt10': round(nb, 4)})
    print(f"  {key:40s}: NB = {nb:+.4f}")
pd.DataFrame(nb_010_records).to_csv(os.path.join(RESULTS_DIR, 'nb_at_pt10.csv'), index=False)

fig, ax = plt.subplots(figsize=(10,7))
styles = {'Treat All':'--', 'Treat None':':'}
for mn in dca_df['model'].unique():
    sub = dca_df[dca_df['model']==mn]
    s = styles.get(mn,'-'); lw = 1 if mn in styles else 2
    ax.plot(sub['threshold'], sub['net_benefit'], s, label=mn, linewidth=lw)
ax.set_xlabel('Threshold Probability'); ax.set_ylabel('Net Benefit')
ax.set_title('Decision Curve Analysis — MIMIC-IV')
ax.legend(loc='upper right', fontsize=8); ax.set_xlim([0,0.5])
ax.axhline(0,color='k',lw=0.5); ax.grid(alpha=0.3)
ax.axvline(0.10, color='red', linestyle=':', alpha=0.5, label='p_t=0.10')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'fig_dca.png'), dpi=300, bbox_inches='tight')
plt.show()
print('\nDCA complete.')

---
## Step 13b: Platt Scaling and Recalibrated DCA (Reviewer II.5)

**Reviewer II.5:** On the ICU test set without recalibration, NEWS2 scoring outperforms CAS-4 hybrid A in net benefit at the clinically relevant 5–10% threshold range — despite the AUROC advantage of CAS-4 hybrid A. The paper claims that a Platt-scaling step "is expected to align" the ML net benefits with the discrimination advantage, but does not demonstrate this empirically.

This cell fits Platt scaling on the **validation partition** (not test) and recomputes DCA on the test partition for CAS-4 hybrid A + XGBoost. It then compares NB at p_t=0.10 before and after recalibration against NEWS2 scoring.

Output: `results/dca_post_platt.csv`, `nb_at_pt10_post_platt.csv`, `figures/fig_dca_post_platt.png`


In [ ]:
# Step 13b: Fit Platt scaling on validation set and recompute DCA on test set.
# This empirically tests the claim that recalibration translates the AUROC
# advantage of CAS-4 hybrid A over NEWS2 into a clinical-utility advantage
# at p_t=0.10.

from sklearn.linear_model import LogisticRegression as _PlattLR

# We need the validation-set predictions of CAS-4 hybrid A to fit Platt
# safely (without using the test set).
required_for_platt = ['CAS-4 hybrid A + XGBoost']
if 'all_val_predictions' not in dir():
    print('⚠ all_val_predictions not in scope. Step 7 must export validation-set')
    print('  predictions to all_val_predictions[key] for this analysis.')
    print('  As a fallback, we will retrain Platt on a 50/50 split of the test set,')
    print('  which is biased but preserves the relative ordering before/after.')
    rng_p = np.random.RandomState(42)
    fallback_idx = rng_p.permutation(len(y_te))
    half = len(fallback_idx) // 2
    fit_idx, eval_idx = fallback_idx[:half], fallback_idx[half:]
    fit_y, eval_y = y_te[fit_idx], y_te[eval_idx]
    p_to_use = all_predictions['CAS-4 hybrid A + XGBoost']
    fit_p, eval_p = p_to_use[fit_idx], p_to_use[eval_idx]
else:
    print('Using validation-set predictions to fit Platt scaling.')
    fit_y = y.loc[val_idx].values
    fit_p = all_val_predictions['CAS-4 hybrid A + XGBoost']
    eval_y = y_te
    eval_p = all_predictions['CAS-4 hybrid A + XGBoost']

# Fit Platt scaling: logistic regression on logit(predicted) -> y
eps = 1e-7
fit_logit = np.log(np.clip(fit_p, eps, 1-eps) / (1 - np.clip(fit_p, eps, 1-eps)))
platt = _PlattLR(fit_intercept=True, max_iter=1000)
platt.fit(fit_logit.reshape(-1, 1), fit_y)
print(f'Platt parameters: a={platt.coef_[0][0]:.4f}, b={platt.intercept_[0]:.4f}')

# Apply Platt to evaluation predictions
eval_logit = np.log(np.clip(eval_p, eps, 1-eps) / (1 - np.clip(eval_p, eps, 1-eps)))
eval_p_platt = platt.predict_proba(eval_logit.reshape(-1, 1))[:, 1]

# Verify recalibration improves slope toward 1.0
print(f'\nCalibration before Platt: slope={cal_slope(eval_y, eval_p):.4f}, '
      f'CITL={cal_itl(eval_y, eval_p):+.4f}')
print(f'Calibration after Platt:  slope={cal_slope(eval_y, eval_p_platt):.4f}, '
      f'CITL={cal_itl(eval_y, eval_p_platt):+.4f}')

# Recompute NB at p_t=0.10 for the recalibrated CAS-4 hybrid A
nb_post = net_benefit(eval_y, eval_p_platt, 0.10)
nb_pre = net_benefit(eval_y, eval_p, 0.10)
print(f'\nNet benefit at p_t=0.10:')
print(f'  CAS-4 hybrid A (pre Platt):   NB = {nb_pre:.4f}')
print(f'  CAS-4 hybrid A (post Platt):  NB = {nb_post:.4f}')

# Compare against NEWS2 scoring on the same eval set
if 'NEWS2 scoring' in all_predictions:
    nb_news2 = net_benefit(eval_y, all_predictions['NEWS2 scoring'][:len(eval_y)] if len(eval_y) < len(y_te) else all_predictions['NEWS2 scoring'], 0.10)
    print(f'  NEWS2 scoring:                NB = {nb_news2:.4f}')
    if nb_post > nb_news2:
        print(f'  → After Platt scaling, CAS-4 hybrid A NB ({nb_post:.4f}) EXCEEDS NEWS2 ({nb_news2:.4f}).')
        print(f'    Platt-adjusted clinical-utility advantage is empirically demonstrated.')
    else:
        print(f'  → After Platt scaling, CAS-4 hybrid A NB ({nb_post:.4f}) still BELOW NEWS2 ({nb_news2:.4f}).')
        print(f'    The discrimination advantage does not translate to clinical utility')
        print(f'    at this threshold even after recalibration. Manuscript claim must be revised.')

# Recompute full DCA curve post-Platt
post_records = []
for pt in thresholds:
    post_records.append({'threshold': pt, 'model': 'CAS-4 hybrid A (Platt)',
                         'net_benefit': net_benefit(eval_y, eval_p_platt, pt)})
post_df = pd.DataFrame(post_records)
out_path = os.path.join(RESULTS_DIR, 'dca_post_platt.csv')
post_df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')


---
## Step 14: Subgroup Analysis

AUROC + calibration by age, sex, ICU type.

In [ ]:
best_key = [k for k in all_predictions if 'CAS-4 hybrid A + XGBoost' in k]
best_key = best_key[0] if best_key else list(all_predictions.keys())[0]
best_prob = all_predictions[best_key]
print(f'Subgroup model: {best_key}')

test_df = features.loc[test_idx].copy().reset_index(drop=True)
if 'age' in test_df.columns:
    test_df['age_group'] = pd.cut(test_df['age'], bins=[0,55,70,200], labels=['<=55','56-70','>70'])

sg_results = []
sg_cols = [c for c in ['age_group','gender'] if c in test_df.columns]
if 'first_careunit' in features.columns:
    sg_cols.append('first_careunit')

for col in sg_cols:
    for grp, gdf in test_df.groupby(col):
        idx = gdf.index.values
        if len(idx) < 50: continue
        yt = y_te[idx]; yp = best_prob[idx]
        if len(np.unique(yt)) < 2: continue
        auroc = roc_auc_score(yt, yp)
        slope = cal_slope(yt, yp)
        sg_results.append({'subgroup':col, 'value':str(grp), 'n':len(idx),
            'events':int(yt.sum()), 'auroc':round(auroc,4), 'cal_slope':round(slope,3)})
        print(f'  {col}={grp}: n={len(idx)}, AUROC={auroc:.4f}, slope={slope:.3f}')

sg_df = pd.DataFrame(sg_results)
sg_df.to_csv(os.path.join(RESULTS_DIR, 'subgroup_results.csv'), index=False)

if len(sg_df) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(sg_df)*0.5)))
    labels = [f"{r['subgroup']}: {r['value']} (n={r['n']})" for _,r in sg_df.iterrows()]
    ax.barh(range(len(sg_df)), sg_df['auroc'], height=0.6, color='#2E86C1', alpha=0.7)
    for j, val in enumerate(sg_df['auroc']):
        ax.text(val+0.003, j, f'{val:.3f}', va='center')
    ax.set_yticks(range(len(sg_df))); ax.set_yticklabels(labels)
    ax.set_xlabel('AUROC'); ax.set_title('Subgroup Analysis')
    ax.grid(alpha=0.3, axis='x'); ax.set_xlim([0.5, 1.0])
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig_subgroups.png'), dpi=300, bbox_inches='tight')
    plt.show()

---
## Step 15: eICU Multi-Center External Validation

**v4.1 hardening:** Step 15 is now 2 monolithic cells (was 7 in v4.0).

- **Cell A** runs the entire eICU pipeline atomically: download →
  cohort filter → features → hourly A(t). Hard verification at the end
  ensures `eicu_feat` has all A(t) features before Cell B runs.
- **Cell B** applies the MIMIC-IV-trained model with hard assertions
  that refuse to proceed if A(t) features are missing or NaN.

This prevents the v4.0 bug where skipping the A(t) sub-cell caused
silent corruption (model fell back to median-imputed A(t) features).

**v4 fix recap:** A(t) for eICU computed on hourly grid matching
MIMIC-IV (was `at_max=at_mean`, `at_std=0` in v3).
**Without retraining or recalibration of the model.**

In [ ]:
# ============================================================
# Step 15 — Cell A: eICU pipeline (download → A(t))
# ============================================================
# This cell is INTENTIONALLY MONOLITHIC. Running it produces a complete
# eicu_feat DataFrame (with A(t) features) or raises an error.
# This prevents the v4.0 bug where a sub-step could be silently skipped.
#
# Phases (announced as we run):
#   1. PhysioNet session
#   2. Download eICU tables (skipped if cached)
#   3. Patient cohort filter (age >=18, LOS >=24h, first stay)
#   4. Vital sign load and plausibility filter
#   5. Hourly grid construction with forward-fill (3h)
#   6. 9 temporal statistical features per vital sign
#   7. Hourly A(t) using train-only weights/pop_stats
#
# Output: eicu_feat with all features INCLUDING at_max, at_mean,
# at_std, at_last. Cell B refuses to apply the model without these.
# ============================================================

print('=' * 60)
print('Step 15: eICU External Validation — Data Preparation')
print('=' * 60)
print('\nPopulation stats (from MIMIC-IV training set):')
for vs in CORE_VITALS:
    print(f"  {vs}: mean={pop_stats[vs]['mean']:.2f}, std={pop_stats[vs]['std']:.2f}")
print(f'\nA(t) weights (train-only): {AT_WEIGHTS}')

# ---- Phase 1: PhysioNet session ----
print('\n[Phase 1/7] PhysioNet session')
_session = globals().get('session')
if _session is None:
    import requests, getpass
    pw_env = os.environ.get('PHYSIONET_PASSWORD')
    password = pw_env if pw_env else getpass.getpass(f'PhysioNet password for {PHYSIONET_USER}: ')
    session = requests.Session()
    login_url = 'https://physionet.org/login/'
    session.get(login_url)
    csrf = session.cookies.get('csrftoken', '')
    resp = session.post(login_url, data={'username': PHYSIONET_USER, 'password': password,
                                          'csrfmiddlewaretoken': csrf},
                        headers={'Referer': login_url})
    print(f'  Session established (status={resp.status_code}).')
else:
    print('  Session reused from earlier.')

# ---- Phase 2: Download ----
print('\n[Phase 2/7] Download eICU tables')
eicu_dir = os.path.join(RAW_DIR, 'eicu-crd')
os.makedirs(eicu_dir, exist_ok=True)

for table in ['patient', 'vitalPeriodic']:
    filename = f'{table}.csv.gz'
    url = f'https://physionet.org/files/eicu-crd/2.0/{filename}'
    filepath = os.path.join(eicu_dir, filename)
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
        print(f'  [SKIP] {filename} ({os.path.getsize(filepath)/1e6:.0f} MB)')
        continue
    print(f'  [GET] {filename}...', end=' ', flush=True)
    resp = session.get(url, stream=True); resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0)); dl = 0
    with open(filepath, 'wb') as f:
        for chunk in resp.iter_content(1024*1024):
            f.write(chunk); dl += len(chunk)
            if total: print(f'\r  [GET] {filename} {dl/1e6:.0f}/{total/1e6:.0f} MB', end='', flush=True)
    print(f'\r  [DONE] {filename} ({os.path.getsize(filepath)/1e6:.0f} MB)' + ' '*20)

# ---- Phase 3: Patient cohort ----
print('\n[Phase 3/7] Patient cohort filter')
eicu_pat = pd.read_csv(os.path.join(eicu_dir, 'patient.csv.gz'), compression='gzip')
print(f'  Raw patient table: {len(eicu_pat)} rows')

eicu_pat['age_num'] = eicu_pat['age'].replace('> 89', '89')
eicu_pat['age_num'] = pd.to_numeric(eicu_pat['age_num'], errors='coerce')
eicu_pat = eicu_pat[eicu_pat['age_num'] >= 18]
eicu_pat = eicu_pat.sort_values('patientunitstayid').drop_duplicates('uniquepid', keep='first')
eicu_pat = eicu_pat[eicu_pat['unitdischargeoffset'] >= 24*60]
eicu_pat['mortality'] = (eicu_pat['hospitaldischargestatus']=='Expired').astype(int)
print(f'  After inclusion criteria: {len(eicu_pat)} stays, '
      f'{eicu_pat["hospitalid"].nunique()} hospitals')

# ---- Phase 4: Vital signs ----
print('\n[Phase 4/7] Vital signs load and plausibility filter')
eicu_vit = pd.read_csv(
    os.path.join(eicu_dir, 'vitalPeriodic.csv.gz'), compression='gzip',
    usecols=['patientunitstayid','observationoffset','heartrate','respiration','sao2','temperature']
)
valid_ids = set(eicu_pat['patientunitstayid'])
eicu_vit = eicu_vit[
    (eicu_vit['patientunitstayid'].isin(valid_ids)) &
    (eicu_vit['observationoffset'] >= 0) &
    (eicu_vit['observationoffset'] <= 24*60)
]
eicu_vit = eicu_vit.rename(columns={'heartrate':'heart_rate','respiration':'resp_rate','sao2':'spo2'})
print(f'  Vital measurements (first 24h): {len(eicu_vit):,}')

for vital_name, (lo, hi) in PLAUSIBLE_RANGES.items():
    if vital_name in eicu_vit.columns:
        bad = (eicu_vit[vital_name] < lo) | (eicu_vit[vital_name] > hi)
        eicu_vit.loc[bad, vital_name] = np.nan

# ---- Phase 5: Hourly grid ----
print('\n[Phase 5/7] Hourly grid construction')
eicu_vit['hour'] = (eicu_vit['observationoffset'] / 60).astype(int).clip(0, 23)
e_hourly_long = eicu_vit.groupby(['patientunitstayid', 'hour'])[CORE_VITALS].mean().reset_index()
e_all_hours = pd.DataFrame({
    'hour': list(range(24)) * len(eicu_pat),
    'patientunitstayid': np.repeat(eicu_pat['patientunitstayid'].values, 24)
})
e_hourly_full = e_all_hours.merge(e_hourly_long, on=['patientunitstayid', 'hour'], how='left')
e_hourly_full = e_hourly_full.sort_values(['patientunitstayid', 'hour'])
for col in CORE_VITALS:
    if col in e_hourly_full.columns:
        e_hourly_full[col] = (
            e_hourly_full.groupby('patientunitstayid')[col]
            .transform(lambda s: s.fillna(method='ffill', limit=3))
        )
print(f'  Hourly grid: {len(e_hourly_full):,} rows')

e_completeness = (
    e_hourly_full.groupby('patientunitstayid')[CORE_VITALS]
    .apply(lambda df: (df.notna().sum() >= 6).all())
)
e_valid = e_completeness[e_completeness].index.tolist()
e_hourly_full = e_hourly_full[e_hourly_full['patientunitstayid'].isin(e_valid)]
print(f'  Stays with >=6h all 4 vitals: {len(e_valid)}')

# ---- Phase 6: Temporal features ----
print('\n[Phase 6/7] Temporal statistical features')
e_feature_rows = []
for sid, group in tqdm(e_hourly_full.groupby('patientunitstayid'), desc='eICU features'):
    row = {'patientunitstayid': sid}
    for vital in CORE_VITALS:
        vals = group[vital].dropna()
        if len(vals) == 0:
            for s in SUFFIXES:
                row[f'{vital}{s}'] = np.nan
            continue
        row[f'{vital}_mean']  = vals.mean()
        row[f'{vital}_std']   = vals.std() if len(vals) > 1 else 0
        row[f'{vital}_min']   = vals.min()
        row[f'{vital}_max']   = vals.max()
        row[f'{vital}_first'] = vals.iloc[0]
        row[f'{vital}_last']  = vals.iloc[-1]
        row[f'{vital}_range'] = vals.max() - vals.min()
        row[f'{vital}_cv']    = (vals.std() / vals.mean()) if vals.mean() != 0 else 0
        if len(vals) >= 2:
            row[f'{vital}_trend'] = np.polyfit(np.arange(len(vals)), vals.values, 1)[0]
        else:
            row[f'{vital}_trend'] = 0.0
    e_feature_rows.append(row)

eicu_feat = pd.DataFrame(e_feature_rows)
eicu_feat = eicu_feat.merge(
    eicu_pat[['patientunitstayid','mortality','age_num','hospitalid']],
    on='patientunitstayid'
)
eicu_feat = eicu_feat.dropna(subset=[f'{vs}_mean' for vs in CORE_VITALS])
print(f'  Feature matrix: {eicu_feat.shape}')

# ---- Phase 7: Hourly A(t) ----
print('\n[Phase 7/7] Hourly A(t) using train-only weights/pop_stats')
# compute_at_hourly auto-detects 'patientunitstayid' for eICU
e_at_df = compute_at_hourly(e_hourly_full, weights, pop_stats, multipliers, CORE_VITALS)
e_at_summary = e_at_df.groupby('patientunitstayid')['at_value'].agg(
    at_max='max', at_mean='mean', at_std='std', at_last='last'
).reset_index()
e_at_summary['at_std'] = e_at_summary['at_std'].fillna(0)

# Defensive: drop any pre-existing A(t) cols and merge fresh
eicu_feat = eicu_feat.drop(columns=[c for c in ['at_max','at_mean','at_std','at_last']
                                      if c in eicu_feat.columns])
eicu_feat = eicu_feat.merge(e_at_summary, on='patientunitstayid', how='left')

# Hard verification before exiting cell
_at_present = [c for c in ['at_max','at_mean','at_std','at_last'] if c in eicu_feat.columns]
assert len(_at_present) == 4, f'A(t) merge failed. Present: {_at_present}'
assert eicu_feat['at_mean'].notna().all(), 'A(t) values contain NaN — merge mismatch'

print(f'\n  A(t) max:  {eicu_feat["at_max"].mean():.3f} +/- {eicu_feat["at_max"].std():.3f}')
print(f'  A(t) std:  {eicu_feat["at_std"].mean():.3f} +/- {eicu_feat["at_std"].std():.3f}')
print(f'  A(t) mean: {eicu_feat["at_mean"].mean():.3f} +/- {eicu_feat["at_mean"].std():.3f}')

# Persist for reproducibility
eicu_feat.to_csv(os.path.join(PROCESSED_DIR, 'eicu_features.csv.gz'),
                 index=False, compression='gzip')

print('\n' + '=' * 60)
print(f'eICU READY: {len(eicu_feat)} stays, {eicu_feat["hospitalid"].nunique()} hospitals')
print(f'  Mortality: {eicu_feat["mortality"].mean():.1%}')
print(f'  A(t) features verified present.')
print(f'  Saved to:  {os.path.join(PROCESSED_DIR, "eicu_features.csv.gz")}')
print('=' * 60)
print('\nProceed to Cell B to apply the MIMIC-IV-trained model.')

In [ ]:
# ============================================================
# Step 15 — Cell B: Apply MIMIC-IV-trained model to eICU
# ============================================================
# Hard guards prevent the v4.0 silent-failure bug:
#   - Refuses to run if eicu_feat is missing (Cell A not executed)
#   - Refuses to run if A(t) features are absent (Cell A interrupted)
#   - Refuses to run if trained model is missing (Step 7 not executed)
#
# This cell does NOT silently fill A(t) features with median.
# ============================================================

# ---- Hard guards ----
assert 'eicu_feat' in dir() and isinstance(eicu_feat, pd.DataFrame), \
    'eicu_feat is missing. Run Step 15 Cell A (data preparation) first.'

_required_at = ['at_max', 'at_mean', 'at_std', 'at_last']
_missing_at = [c for c in _required_at if c not in eicu_feat.columns]
assert not _missing_at, \
    f'A(t) features missing from eicu_feat: {_missing_at}. Re-run Step 15 Cell A.'
assert eicu_feat['at_mean'].notna().all(), \
    'A(t) values contain NaN. Re-run Step 15 Cell A.'

assert 'xgb_models' in dir() and 'CAS-4 hybrid A' in xgb_models, \
    'Trained CAS-4 hybrid A model not found in xgb_models. Re-run Step 7.'

print('All guards passed.')
print(f'  eicu_feat shape: {eicu_feat.shape}')
print(f'  A(t) features:   {_required_at} — all present, no NaN')

cas4h_model, cas4h_scaler, cas4h_cols = xgb_models['CAS-4 hybrid A']
avail_e = [c for c in cas4h_cols if c in eicu_feat.columns]
non_at_missing = [c for c in cas4h_cols if c not in eicu_feat.columns]

print(f'\n  Model expects {len(cas4h_cols)} columns')
print(f'  Available in eICU: {len(avail_e)}')
if non_at_missing:
    print(f'  Missing in eICU (filled with 0 = scaled mean): {non_at_missing}')
    # These should only be non-CORE vitals (e.g. SBP, GCS) that eICU lacks.
    # Filling with 0 (= scaled mean) represents "feature not measured".
    # A(t) features are guaranteed present by guards above.

# Build matrix in original column order (critical for the trained scaler)
X_eicu_raw = pd.DataFrame(index=eicu_feat.index)
for c in cas4h_cols:
    X_eicu_raw[c] = eicu_feat[c] if c in eicu_feat.columns else 0.0
X_eicu_raw = X_eicu_raw.fillna(0)
X_e = cas4h_scaler.transform(X_eicu_raw.values)

y_eicu = eicu_feat['mortality'].values
y_prob_eicu = cas4h_model.predict_proba(X_e)[:, 1]

# Metrics
auroc_e = roc_auc_score(y_eicu, y_prob_eicu)
lo_e, hi_e = bootstrap_ci(y_eicu, y_prob_eicu)
slope_e = cal_slope(y_eicu, y_prob_eicu)
brier_e = brier_score_loss(y_eicu, y_prob_eicu)

print('\n' + '=' * 60)
print('eICU CAS-4 hybrid A + XGBoost (v4.1: hourly A(t), no leakage):')
print('=' * 60)
print(f'  AUROC:        {auroc_e:.4f} ({lo_e:.3f}-{hi_e:.3f})')
print(f'  Cal. slope:   {slope_e:.3f}')
print(f'  Brier score:  {brier_e:.4f}')
print(f'  N:            {len(eicu_feat)}')
print(f'  Hospitals:    {eicu_feat["hospitalid"].nunique()}')
print(f'  Mortality:    {y_eicu.mean():.1%}')

# Calibration plot
fig, ax = plt.subplots(figsize=(8,8))
frac, mp = calibration_curve(y_eicu, y_prob_eicu, n_bins=10, strategy='quantile')
ax.plot(mp, frac, 's-', color='#2E86C1', linewidth=2, label='CAS-4 hybrid A + XGBoost')
ax.plot([0,1],[0,1],'k--', label='Perfect')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('Observed proportion')
ax.set_title(f'Calibration — eICU ({eicu_feat["hospitalid"].nunique()} hospitals)')
ax.legend(); ax.set_xlim([0,1]); ax.set_ylim([0,1]); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'fig_calibration_eicu.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f'\n  Saved: {os.path.join(FIGURES_DIR, "fig_calibration_eicu.png")}')

---
## Step 15b: Excluded eICU Hospitals — Selection Bias Quantification (Reviewer V.18)

**Reviewer V.18:** The eICU validation included **138 of 208 hospitals** in the database. The 70 excluded hospitals may differ systematically from included ones (e.g., smaller rural hospitals filtered out by the LOS≥24h or "all four core vitals available" criteria), which could bias external-validation results. This cell quantifies the direction of selection bias.


In [ ]:
# Step 15b: Compare included vs. excluded eICU hospitals on key characteristics.
# Inputs (must exist in scope after Step 15 Cell A):
#   eicu_patient_full   - DataFrame of ALL eICU patients before inclusion filter
#   eicu_feat           - DataFrame of patients meeting all inclusion criteria
# If these names differ in your pipeline, adjust the variable references below.

# Try common variable names used by Step 15
candidates_full = ['eicu_patient_full', 'eicu_patient_raw', 'eicu_pat_all', 'patient_eicu_all']
candidates_inc  = ['eicu_feat', 'eicu_features', 'eicu_inc']

src_full = next((n for n in candidates_full if n in dir()), None)
src_inc  = next((n for n in candidates_inc  if n in dir()), None)

if src_full is None or src_inc is None:
    print('⚠ Could not auto-locate full/included eICU patient frames in scope.')
    print(f'  Candidates checked (full): {candidates_full}')
    print(f'  Candidates checked (inc):  {candidates_inc}')
    print('  Manual fix: set df_full and df_inc below to your DataFrames')
    print('  containing all eICU patients (df_full) and the included subset (df_inc),')
    print('  each with at least columns: hospitalid, hospitaldischargestatus, age, unitdischargeoffset.')
else:
    df_full = globals()[src_full]
    df_inc = globals()[src_inc]
    print(f'Using {src_full} (n={len(df_full)}) as full cohort, {src_inc} (n={len(df_inc)}) as included.')

    # Determine hospital ID column
    hosp_col = next((c for c in ['hospitalid', 'hospital_id', 'site_id'] if c in df_full.columns), None)
    if hosp_col is None:
        print(f'⚠ No hospital ID column in {src_full}. Columns: {list(df_full.columns)[:30]}')
    else:
        included_hosp = set(df_inc[hosp_col].dropna().unique())
        all_hosp = set(df_full[hosp_col].dropna().unique())
        excluded_hosp = all_hosp - included_hosp
        print(f'\nTotal hospitals in full eICU patient table: {len(all_hosp)}')
        print(f'Included after filters:                       {len(included_hosp)}')
        print(f'Excluded:                                     {len(excluded_hosp)}')

        # Per-hospital summary
        def hospital_summary(df, hosp_set, label):
            sub = df[df[hosp_col].isin(hosp_set)]
            stats = []
            for h in hosp_set:
                hdf = sub[sub[hosp_col] == h]
                stats.append({
                    'group': label, 'hospital_id': h,
                    'n_stays': len(hdf),
                    'mortality': float(hdf['hospitaldischargestatus'].eq('Expired').mean())
                                 if 'hospitaldischargestatus' in hdf.columns else None,
                    'median_age': float(hdf['age'].median()) if 'age' in hdf.columns else None,
                    'median_los_min': float(hdf['unitdischargeoffset'].median())
                                      if 'unitdischargeoffset' in hdf.columns else None,
                })
            return pd.DataFrame(stats)

        inc_stats = hospital_summary(df_full, included_hosp, 'included')
        exc_stats = hospital_summary(df_full, excluded_hosp, 'excluded')
        all_stats = pd.concat([inc_stats, exc_stats], ignore_index=True)

        print('\nGroup comparison (median per-hospital):')
        agg = all_stats.groupby('group').agg({
            'n_stays': 'median', 'mortality': 'median',
            'median_age': 'median', 'median_los_min': 'median'
        }).round(3)
        print(agg)

        out_path = os.path.join(RESULTS_DIR, 'eicu_hospital_selection_bias.csv')
        all_stats.to_csv(out_path, index=False)
        print(f'\nSaved: {out_path}')


---
## Step 16: Save All Results to JSON

Master JSON for Word template placeholders.

In [ ]:
from datetime import datetime
from pathlib import Path

RESULTS_FINAL = {
    'metadata': {
        'generated_at': datetime.now().isoformat(),
        'version': 'Scenario_C_v5_gcs_fix',
        'changes_from_v4': [
            'GCS reconstruction from MIMIC-IV component itemids 220739/223900/223901 (was: legacy itemid 198 from MIMIC-III, which returns no rows in MIMIC-IV chartevents)',
            'Cell 20 feature engineering iterates over all hourly_full vital columns (picks up reconstructed gcs_total)',
            'New L3-NEWS2 SHAP cell: shap_ranking_l3_news2.csv and fig_shap_l3_news2.png',
        ],
        'changes_from_v3': [
            'A(t) weights and pop_stats computed on training set only (no leakage)',
            'eICU A(t) computed hourly (matching MIMIC-IV methodology)',
            'Bootstrap permutation test executed in pipeline',
            'SHAP ranking exported to CSV',
            'DCA NB at p_t=0.10 explicitly reported',
        ],
    },
    'cohorts': {
        'mimic_iv': {
            'n': int(len(cohort)),
            'mortality_n': int(cohort['in_hospital_mortality'].sum()),
            'mortality_rate': round(float(cohort['in_hospital_mortality'].mean()), 3),
            'median_age': float(cohort['age'].median()) if 'age' in cohort.columns else None,
        },
    },
    'split': {
        'train_n': int(len(train_idx)),
        'val_n': int(len(val_idx)),
        'test_n': int(len(test_idx)),
    },
    'at_calibration': {
        'weights': weights,
        'multipliers': multipliers,
        'pop_stats': {k: {'mean': v['mean'], 'std': v['std']} for k, v in pop_stats.items()},
        'computed_on': 'training_set_only',
    },
    'feature_reduction': red_df.to_dict(orient='records'),
    'model_comparison': results_df.to_dict(orient='records') if 'results_df' in dir() else all_results,
    'calibration': cal_df.to_dict(orient='records'),
    'subgroups': sg_results,
}

if 'shap_importance' in dir():
    RESULTS_FINAL['shap_top15'] = shap_importance.head(15).to_dict(orient='records')

if 'nb_010_records' in dir():
    RESULTS_FINAL['dca_nb_at_pt10'] = nb_010_records

if 'eicu_feat' in dir():
    RESULTS_FINAL['cohorts']['eicu'] = {
        'n': int(len(eicu_feat)),
        'n_hospitals': int(eicu_feat['hospitalid'].nunique()),
        'mortality_rate': round(float(eicu_feat['mortality'].mean()), 3),
    }
if 'auroc_e' in dir():
    RESULTS_FINAL['eicu_validation'] = {
        'auroc': round(float(auroc_e), 4),
        'ci_low': round(float(lo_e), 4),
        'ci_high': round(float(hi_e), 4),
        'cal_slope': round(float(slope_e), 3),
        'brier': round(float(brier_e), 4),
    }

def convert(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    elif isinstance(obj, (np.floating,)): return float(obj)
    elif isinstance(obj, np.ndarray): return obj.tolist()
    return obj

json_path = os.path.join(RESULTS_DIR, 'all_results.json')
with open(json_path, 'w') as f:
    json.dump(RESULTS_FINAL, f, indent=2, default=convert)

print(f'JSON saved: {json_path}')
print()
print('ALL FILES:')
for d in [RESULTS_DIR, FIGURES_DIR]:
    if os.path.exists(d):
        for fp in sorted(Path(d).glob('*')):
            print(f'  {fp.name:45s} {fp.stat().st_size/1024:.1f} KB')
print('\n=== STEPS 0-16 COMPLETE ===')
print('Continue to Step 17 (bootstrap permutation test).')

---
## Step 17: Bootstrap Pairwise AUROC Comparison

Statistical comparison between key sensor configurations to confirm the knee point.
**This must be executed for the paper** — provides p-values cited in the abstract
and central findings.

n=2000 bootstrap iterations, RandomState=42 for reproducibility.

In [ ]:
# Bootstrap pairwise AUROC comparison (n=2000, seed=42)
y_te = y.loc[test_idx].values

print('Available predictions from reduction curve:')
for k in sorted(reduction_predictions.keys()):
    auroc = roc_auc_score(y_te, reduction_predictions[k])
    print(f'  {k:18s}: AUROC = {auroc:.4f}')

print('\nAvailable predictions from model comparison:')
for k in sorted(all_predictions.keys()):
    auroc = roc_auc_score(y_te, all_predictions[k])
    print(f'  {k:40s}: AUROC = {auroc:.4f}')

comparisons = [
    ('L5-CAS4',     'L6-3vitals', 'Knee point: 4 vs 3 vitals (L5 vs L6)'),
    ('L6-3vitals',  'L7-2vitals', 'Knee point: 3 vs 2 vitals (L6 vs L7) [REVIEWER II.2 - central claim]'),
    ('L5-CAS4',     'L1-Full',    'CAS-4 vs Full'),
    ('L5b-CAS4+At', 'L5-CAS4',    'A(t) benefit'),
    ('L5-CAS4',     'L3-NEWS2',   'CAS-4 vs NEWS2-set'),
]

n_boot = 2000
rng = np.random.RandomState(42)
bootstrap_records = []

print('\n' + '=' * 70)
print('PAIRWISE AUROC COMPARISONS (Bootstrap Permutation, n=2000)')
print('=' * 70)

for key_a, key_b, desc in comparisons:
    if key_a not in reduction_predictions or key_b not in reduction_predictions:
        print(f'\n  {desc}: SKIPPED ({key_a} or {key_b} not found)')
        continue

    pa = reduction_predictions[key_a]
    pb = reduction_predictions[key_b]
    obs_diff = roc_auc_score(y_te, pa) - roc_auc_score(y_te, pb)

    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_te), len(y_te), replace=True)
        if len(np.unique(y_te[idx])) < 2:
            continue
        try:
            a = roc_auc_score(y_te[idx], pa[idx])
            b = roc_auc_score(y_te[idx], pb[idx])
            diffs.append(a - b)
        except Exception:
            continue

    diffs = np.array(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])

    if obs_diff > 0:
        p_val = float(np.mean(diffs <= 0) * 2)
    elif obs_diff < 0:
        p_val = float(np.mean(diffs >= 0) * 2)
    else:
        p_val = 1.0
    p_val = min(p_val, 1.0)

    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

    print(f'\n  {desc}:')
    print(f'    {key_a} vs {key_b}')
    print(f'    delta_AUROC = {obs_diff:+.4f} (95% CI: {ci_lo:+.4f} to {ci_hi:+.4f})')
    print(f'    p = {p_val:.6f} {sig}')

    bootstrap_records.append({
        'comparison': desc,
        'key_a': key_a, 'key_b': key_b,
        'auroc_a': round(float(roc_auc_score(y_te, pa)), 4),
        'auroc_b': round(float(roc_auc_score(y_te, pb)), 4),
        'delta_auroc': round(float(obs_diff), 4),
        'ci_low': round(float(ci_lo), 4),
        'ci_high': round(float(ci_hi), 4),
        'p_value': round(p_val, 6),
        'significance': sig,
    })

# Model comparisons
print('\n' + '-' * 70)
print('MODEL COMPARISONS (from Step 10)')
print('-' * 70)

model_comparisons = [
    ('CAS-4 hybrid A + XGBoost', 'CAS-4 + XGBoost', 'A(t) benefit (model comparison)'),
    ('CAS-4 + XGBoost',          'NEWS2 scoring',   'CAS-4 XGB vs NEWS2 scoring'),
]

for key_a, key_b, desc in model_comparisons:
    if key_a not in all_predictions or key_b not in all_predictions:
        print(f'\n  {desc}: SKIPPED ({key_a} or {key_b} not found)')
        continue

    pa = all_predictions[key_a]
    pb = all_predictions[key_b]
    obs_diff = roc_auc_score(y_te, pa) - roc_auc_score(y_te, pb)

    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_te), len(y_te), replace=True)
        if len(np.unique(y_te[idx])) < 2:
            continue
        try:
            diffs.append(roc_auc_score(y_te[idx], pa[idx]) - roc_auc_score(y_te[idx], pb[idx]))
        except Exception:
            continue

    diffs = np.array(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    if obs_diff > 0:
        p_val = float(np.mean(diffs <= 0) * 2)
    elif obs_diff < 0:
        p_val = float(np.mean(diffs >= 0) * 2)
    else:
        p_val = 1.0
    p_val = min(p_val, 1.0)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

    print(f'\n  {desc}:')
    print(f'    {key_a} vs {key_b}')
    print(f'    delta_AUROC = {obs_diff:+.4f} (95% CI: {ci_lo:+.4f} to {ci_hi:+.4f})')
    print(f'    p = {p_val:.6f} {sig}')

    bootstrap_records.append({
        'comparison': desc,
        'key_a': key_a, 'key_b': key_b,
        'auroc_a': round(float(roc_auc_score(y_te, pa)), 4),
        'auroc_b': round(float(roc_auc_score(y_te, pb)), 4),
        'delta_auroc': round(float(obs_diff), 4),
        'ci_low': round(float(ci_lo), 4),
        'ci_high': round(float(ci_hi), 4),
        'p_value': round(p_val, 6),
        'significance': sig,
    })

# Save bootstrap results
boot_df = pd.DataFrame(bootstrap_records)
boot_df.to_csv(os.path.join(RESULTS_DIR, 'bootstrap_auroc_comparisons.csv'), index=False)
print(f'\nSaved: {os.path.join(RESULTS_DIR, "bootstrap_auroc_comparisons.csv")}')

# Append to master JSON
json_path = os.path.join(RESULTS_DIR, 'all_results.json')
if os.path.exists(json_path):
    with open(json_path) as f:
        results_final = json.load(f)
    results_final['bootstrap_comparisons'] = bootstrap_records
    with open(json_path, 'w') as f:
        json.dump(results_final, f, indent=2, default=convert)
    print(f'Updated: {json_path}')

print('\n' + '=' * 70)
print('PIPELINE COMPLETE — use these p-values in the paper.')
print('=' * 70)